# Bezout-Schur + 5-Word QROM

**26 October 2026 · source-built chemistry · CPU reproduction**

**Use spare lookup qubits to load five angles at once.** The chemistry family shrinks from **19 circuits to 15**. Each circuit uses 123 qubits, below the 128-qubit cap. The complete chemistry cost falls from **432.325 million to 426.974 million: 1.24% lower**, with **identical factorization, angle words and physical errors**.

The portfolio combines mrmeows's 63-qubit inverse square root, Ahmed Berat Özer's new Bézout–Schur decoder, and his R4/B255/C128 factorization recompiled here at J=5. The decoder and factors are credited public components. The new contribution here is the wider lookup schedule, its kernel-relation budget, exact source rebuild and controlled full-cost comparison.

**Fixed-salt full-verifier projection: 19.599953.** The previous archive received **19.24359** on the public leaderboard. The observed score for this notebook is shown in Kaggle's Best Score badge; the fixed-salt projection is a separate measurement.

**Run All** reconstructs the chemistry circuits from source and saved algebraic plans, runs correctness checks and writes one `submission.zip`. No GPU or compilation search is needed for this path. The earlier C59 compression study remains in the notebook history and is discussed below.

## 1. Rebuild and verify

Attach the competition, [Ahmed's v8 components, version 1](https://www.kaggle.com/datasets/ahmedberatozer/vqcc-v8-components/versions/1), and [Alex Chilton's output, version 37](https://www.kaggle.com/code/alexchilton/toffolis-go-18-3-schur-dqi-at-883-qubits), which supplies the unchanged 63-qubit invsqrt binary. All input and generated-file hashes are checked.

The compiler uses saved seeds and low-monomial permutations to regenerate exact GF(2) decomposition certificates. It checks the tensor reconstruction and emits the gates with the credited CircuitBuilder compiler. Saved plans avoid repeating the offline search. Full receipts are reused only for identical bytes, reference/verifier/native fingerprints, runtime and development salt; chemistry and independent invsqrt/DQI probes are checked afresh. Set `FORCE_FULL = True` in the second code cell to repeat every full simulation.

In [ ]:
import os
os.environ.update(OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1')
import gc,hashlib,json,lzma,math,shutil,sys,time,zipfile
from pathlib import Path
import numpy as np
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working')
COMP=next(p.parents[2] for p in INPUT.rglob('verify_submission_invsqrt.py'))
sys.path[:0]=[str(COMP/'kickmix'),str(COMP/'verifiers')]
import kickmix as km
from invsqrt import verify_submission_invsqrt as IV
from dqi import verify_submission_dqi as DV
from dfthcsos.verify import verify as verify_chemistry
FINAL=Path('/tmp/wide-qrom-selected');FINAL.mkdir(exist_ok=True)
(FINAL/'dfthcsos').mkdir(exist_ok=True)
WORKERS=min(4,os.cpu_count() or 1)
# Retained from the earlier release, not selected by a failure count.
SALT='lightningv08-v13-fixed-20261026'
REFERENCE={'invsqrt':2076774.233715282,'dqi':24439473914136.668,'dfthcsos':3355599588.3155365}
PIN=json.loads(r'''{"members":{"dfthcsos/angle_k_0.kmb":"b631c3cafd2edc600edeed997f28c1b1b8a71234189bcc7d6a3afdf4a47408db","dfthcsos/angle_k_1.kmb":"974dc12398186f359e79d38d733720c5231686d7f591dcb1ac1b5d20be27f47e","dfthcsos/angle_k_10.kmb":"ac3cdb0d8d742d9629d3d5e3d5acb843012a084557587fe2c0ad9b9058da06d5","dfthcsos/angle_k_11.kmb":"9b103b5aadfc52e1b481894dff426dbf7443b5a3530db2c1e325ef6a39ba63e2","dfthcsos/angle_k_12.kmb":"d872682a1672429fdbbb44f56dbac9f4a0e951231817926d5470ff36f64aac3d","dfthcsos/angle_k_13.kmb":"42044bcb48d80947ef2900aea4eb2eaa8593f1b809d37bada48f1ab169b44ae3","dfthcsos/angle_k_14.kmb":"f3bb8ce79e39ecddad5bb1ff57e6818158c4cba1133b54f0009f1133b8585d49","dfthcsos/angle_k_2.kmb":"e25d78895511c50babe940b21314ede6ad79794d38a9da53efaefbdc802864ac","dfthcsos/angle_k_3.kmb":"ac59a43cf75a516937ad12900916c292add65558902df77b18d0600cc45a7ecb","dfthcsos/angle_k_4.kmb":"64d9dfc71e0fb4c5c55e4a01ba9063c579d17d2602551a0899253f8edd1dc136","dfthcsos/angle_k_5.kmb":"6c4b1e3a59e46a604528f80912f4ccfd9ce689c0f39cd548521d0663d9d1821a","dfthcsos/angle_k_6.kmb":"9932019836bbe8217a8fd34d1b54bbaccf9591d1c83f56c2a0cb20e2e14a2c7d","dfthcsos/angle_k_7.kmb":"7f1444666cf99cac8533089c862edfeb60db5e41daedf693c03af0d0be4b4392","dfthcsos/angle_k_8.kmb":"d748449d2c77cf858bdd90d69054bc3a3ac1d81ad2501c21d85e5b04a824e66b","dfthcsos/angle_k_9.kmb":"039ec2a8c81ff647a83832dc76a687947a80f81202361b9af2f1c6cef1e29268","dfthcsos/submission.npz":"a5eabd76c2b2869e39511a53130cd909034dec6f26fa3db61354a9d12ffe6413","dqi.kmb":"569863ae0fafd5f94fe91b498ec67ed0a4d96b826ced31cb0ff57c7fe6c063cb","invsqrt.kmb":"9ac50386192a544f353ab0f46604e5e072981b8bfc8521e8551c689a9445e3b9"},"dqi_xz_sha256":"e09106a0dd08b9f1b8f355947a3b208775a1d4b9c38cb39784a8a37038acb311","invsqrt_source_archive":"d4e76cb5ccadf4688115e4a60ccc987cf2babc95046a1b3bc60ed0a05ae10fd6"}''')
PLANS=json.loads(r'''[{"k":0,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":7,"permutation":[9,14,1,3,8,10,0,13,11,6,7,4,12,2,5]},{"k":1,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":5,"permutation":[9,0,1,3,13,2,8,12,14,6,5,10,7,11,4]},{"k":2,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":7,"permutation":[2,0,13,9,5,8,3,7,1,10,14,4,11,6,12]},{"k":3,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":4,"permutation":[8,4,10,2,1,0,9,13,14,5,7,3,12,6,11]},{"k":4,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":4,"permutation":[13,1,7,3,5,0,4,11,9,12,8,14,2,6,10]},{"k":5,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":0,"permutation":[9,13,14,12,1,0,11,4,3,2,7,8,5,10,6]},{"k":6,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":7,"permutation":[0,2,10,11,1,6,12,7,3,5,9,8,13,14,4]},{"k":7,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":4,"permutation":[5,10,1,0,9,3,7,6,11,2,13,8,14,12,4]},{"k":8,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":5,"permutation":[13,2,7,6,12,3,0,9,11,4,14,5,1,10,8]},{"k":9,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":0,"permutation":[7,5,14,1,12,10,9,0,3,8,2,13,4,6,11]},{"k":10,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":2,"permutation":[12,8,10,2,4,1,13,6,14,0,7,9,11,3,5]},{"k":11,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":1,"permutation":[10,14,5,1,3,6,4,13,2,7,0,9,12,8,11]},{"k":12,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":7,"permutation":[7,6,14,11,8,9,2,10,12,5,13,3,0,4,1]},{"k":13,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":5,"permutation":[14,11,10,12,0,13,8,2,1,5,6,3,9,4,7]},{"k":14,"T":297.0,"Q":123,"target":450,"paired":225,"relations":75,"rel_base":4,"extra":15,"seed":1,"permutation":[13,10,3,7,2,11,4,0,8,12,14,6,5,9,1]}]''')
COMPILER_SOURCE=json.loads(r'''{"rank_parity.py":"#!/usr/bin/env python3\n\"\"\"Independent coherent lookup via an even-rank output-function basis.\n\nAll emitted gates are unconditional X/CX/CCX. No decomposition certificate or\ncandidate data is imported. The optional organizer subset is a circuit fixture,\nnot a chemistry submission. A rank-one matrix count is never reported as the\nnative paired-product count.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\n\nfrom borrowed_prefix import delta_batch, native_check, symbolic_check\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef set_bits(value):\n    while value:\n        bit = (value & -value).bit_length()-1\n        yield bit\n        value &= value-1\n\n\ndef mobius_axis(values, axis):\n    out = np.swapaxes(values.copy(),axis,0)\n    ids = np.arange(len(out))\n    for bit in range((len(out)-1).bit_length()):\n        rows = ids[(ids & (1 << bit)) != 0]\n        out[rows] ^= out[rows ^ (1 << bit)]\n    return np.swapaxes(out,0,axis)\n\n\ndef canonical_gauge(words, edges, brot):\n    \"\"\"Exact integer projective gauge; all cosines use a positive half-turn.\"\"\"\n    out = np.asarray(words,dtype=np.int64).copy()\n    signs = np.ones((len(words),len(edges)+1),dtype=np.int8)\n    modulus = 1 << brot\n    for k in reversed(range(len(edges))):\n        p,q = map(int,edges[k])\n        x = out[:,k].copy()\n        opposite = signs[:,p] != signs[:,q]\n        x[opposite] = (-x[opposite]) % modulus\n        flip = (x >= modulus//4) & (x < 3*modulus//4)\n        x[flip] = (x[flip]+modulus//2) % modulus\n        signs[flip,p] *= -1\n        signs[:,q] = 1\n        out[:,k] = x\n    assert not np.any(((out >> (brot-1)) ^ (out >> (brot-2))) & 1)\n    return out, signs[:,0]\n\n\ndef decode(words,edges,brot):\n    v = np.zeros((len(words),len(edges)+1))\n    v[:,0] = 1\n    for k,(p,q) in enumerate(edges):\n        theta = words[:,k]*(np.pi/(1 << (brot-1)))\n        left,right = v[:,p].copy(),v[:,q].copy()\n        v[:,p] = np.cos(theta)*left - np.sin(theta)*right\n        v[:,q] = np.sin(theta)*left + np.cos(theta)*right\n    return v\n\n\nclass BinaryBasis:\n    def __init__(self):\n        self.pivots = {}\n        self.vectors = []\n\n    def add(self,value):\n        original = value\n        coordinates = 1 << len(self.vectors)\n        while value:\n            p = value.bit_length()-1\n            if p not in self.pivots:\n                self.pivots[p] = (value,coordinates)\n                self.vectors.append(original)\n                return True\n            v,c = self.pivots[p]\n            value ^= v\n            coordinates ^= c\n        return False\n\n    def solve(self,value):\n        coordinates = 0\n        while value:\n            p = value.bit_length()-1\n            if p not in self.pivots:\n                return None\n            v,c = self.pivots[p]\n            value ^= v\n            coordinates ^= c\n        return coordinates\n\n\ndef nullspace(rows, width):\n    pivots = {}\n    for value in rows:\n        value = int(value)\n        while value:\n            p = value.bit_length()-1\n            if p not in pivots:\n                pivots[p] = value\n                if len(pivots) == width:\n                    return []\n                break\n            value ^= pivots[p]\n    result = []\n    for j in range(width):\n        if j not in pivots:\n            value = 1 << j\n            for p,row in sorted(pivots.items()):\n                if (row & value).bit_count() & 1:\n                    value ^= 1 << p\n            result.append(value)\n    return result\n\n\ndef factor_matrix(columns):\n    basis = BinaryBasis()\n    for col in columns:\n        basis.add(col)\n    lows = [0]*len(basis.vectors)\n    for k,col in enumerate(columns):\n        coordinates = basis.solve(col)\n        assert coordinates is not None\n        for bit in set_bits(coordinates):\n            lows[bit] ^= 1 << k\n    return list(zip(basis.vectors,lows))\n\n\ndef require_redundant_sign_bit(delta,brot):\n    \"\"\"Reject raw tables that cannot be represented by b-1 Boolean functions.\n\n    Every physical word bit remains an arbitrary target. This precondition is\n    about the requested XOR values, not an assumption about target contents.\n    The caller must explicitly apply a legal projective gauge beforehand.\n    \"\"\"\n    if not isinstance(brot,(int,np.integer)) or not 2 <= brot <= 62:\n        raise ValueError('brot must be an integer between 2 and 62')\n    table = np.asarray(delta)\n    if (table.ndim != 2 or min(table.shape,default=0) < 1 or\n            not np.issubdtype(table.dtype,np.integer)):\n        raise ValueError('delta must be a nonempty two-dimensional integer table')\n    if np.any(table < 0) or np.any(table >= (1 << brot)):\n        raise ValueError('delta word outside the selected precision')\n    table = table.astype(np.int64,copy=False)\n    if np.any(((table >> (brot-1)) ^ (table >> (brot-2))) & 1):\n        raise ValueError('redundant sign-bit precondition failed: the two highest '\n                         'bits of every requested XOR word must be equal')\n    return table\n\n\ndef coefficients(delta,brot,lowbits):\n    delta = require_redundant_sign_bit(delta,brot)\n    count,J = delta.shape\n    low = 1 << lowbits\n    high = (count+low-1)//low\n    padded = np.zeros((low*high,J),dtype=np.int64)\n    padded[:count] = delta\n    require_redundant_sign_bit(padded,brot)\n    bits = np.stack([(padded[:,j] >> bit)&1 for j in range(J)\n                     for bit in range(brot-1)],axis=1).astype(np.uint8)\n    truth = bits.reshape(high,low,-1)\n    result = mobius_axis(mobius_axis(truth,0),1)\n    assert np.array_equal(mobius_axis(mobius_axis(result,1),0),truth)\n    return result\n\n\ndef plan(delta,brot,lowbits,max_kernels=32767):\n    ac = coefficients(delta,brot,lowbits)\n    tensor = ac[1:,1:,:]\n    H,L,O = tensor.shape\n    columns = [[sum(int(tensor[h,k,j]) << h for h in range(H))\n                for k in range(L)] for j in range(O)]\n    original_rank = [len(factor_matrix(cols)) for cols in columns]\n    chosen = BinaryBasis()\n    certificates = []\n    # Preserve existing already-even or unusually-low rank directions first.\n    for j,r in sorted(enumerate(original_rank),key=lambda item:(item[1]+1)//2):\n        if r < min(H,L) or r % 2 == 0:\n            if chosen.add(1 << j):\n                certificates.append({'direction':1 << j,'kernel':None,'rank':r})\n    packed_low = [[sum(int(tensor[h,k,j]) << j for j in range(O))\n                   for h in range(H)] for k in range(L)]\n    current = [0]*H\n    previous = 0\n    searched = 0\n    for counter in range(1,min((1 << L)-1,max_kernels)+1):\n        mask = counter ^ (counter >> 1)\n        changed = (mask ^ previous).bit_length()-1\n        current = [a ^ b for a,b in zip(current,packed_low[changed])]\n        previous = mask\n        searched += 1\n        for direction in nullspace(current,O):\n            if chosen.solve(direction) is not None:\n                continue\n            mixture = [0]*L\n            for j in set_bits(direction):\n                mixture = [a ^ b for a,b in zip(mixture,columns[j])]\n            rank = len(factor_matrix(mixture))\n            # A rank drop must actually cross a paired-product threshold.\n            if (rank+1)//2 < (min(H,L)+1)//2:\n                assert chosen.add(direction)\n                certificates.append({'direction':direction,'kernel':mask,'rank':rank})\n        if len(chosen.vectors) == O:\n            break\n    for j in range(O):\n        if chosen.add(1 << j):\n            certificates.append({'direction':1 << j,'kernel':None,\n                                 'rank':original_rank[j]})\n    assert len(chosen.vectors) == O\n    output_directions = [0]*O\n    for j in range(O):\n        inverse_column = chosen.solve(1 << j)\n        assert inverse_column is not None\n        for i in set_bits(inverse_column):\n            output_directions[i] |= 1 << j\n    terms = []\n    reconstructed = [[0]*L for _ in range(O)]\n    products = 0\n    for i,vector in enumerate(chosen.vectors):\n        mixture = [0]*L\n        for j in set_bits(vector):\n            mixture = [a ^ b for a,b in zip(mixture,columns[j])]\n        factors = factor_matrix(mixture)\n        assert len(factors) == certificates[i]['rank']\n        terms.append({'input_output_basis':vector,\n                      'target_direction':output_directions[i],\n                      'factors':factors})\n        products += (len(factors)+1)//2\n        for j in set_bits(output_directions[i]):\n            reconstructed[j] = [a ^ b for a,b in zip(reconstructed[j],mixture)]\n    assert reconstructed == columns\n    return ac,terms,{'H':H,'L':L,'O':O,'searched_nonzero_low_kernels':searched,\n        'original_matrix_ranks':original_rank,'output_basis_certificates':certificates,\n        'untransformed_paired_products':sum((r+1)//2 for r in original_rank),\n        'transformed_paired_products':products,'exact_mixed_tensor_reconstruction':True}\n\n\ndef build(delta,brot,lowbits,ac,terms):\n    import kickmix as km\n    delta = require_redundant_sign_bit(delta,brot)\n    D,J = delta.shape\n    n = (D-1).bit_length()\n    A,B,O = ac.shape\n    H,L = A-1,B-1\n    builder = km.CircuitBuilder()\n    qubits = list(builder.create_quantum_register(n,'selection'))\n    for j in range(J):\n        qubits += list(builder.create_quantum_register(brot,f'word_{j}'))\n    trace = []\n    def gate(name,*ids):\n        trace.append((name,*ids))\n        getattr(builder,name.lower())(*[qubits[i] for i in ids])\n    cache = []\n    def cached_basis(size,offset):\n        result = {0:None}\n        for mask in range(1,size):\n            bit = mask.bit_length()-1\n            if mask.bit_count()==1:\n                result[mask] = offset+bit\n            else:\n                parent = mask ^ (1 << bit)\n                q = len(qubits)\n                qubits.extend(builder.alloc_qubits(1))\n                result[mask] = q\n                triple = (result[parent],offset+bit,q)\n                gate('CCX',*triple)\n                cache.append(triple)\n        return result\n    high = cached_basis(A,lowbits)\n    low = cached_basis(B,0)\n    if len(qubits)>128:\n        raise ValueError(f'cache layout needs {len(qubits)} qubits')\n    basis = [high[i] for i in range(1,A)]+[low[i] for i in range(1,B)]\n    groups = [[n+j*brot+bit] + ([n+j*brot+brot-1] if bit==brot-2 else [])\n              for j in range(J) for bit in range(brot-1)]\n    def target_bits(direction):\n        return [q for j in set_bits(direction) for q in groups[j]]\n    def linear_output(control,direction):\n        for q in target_bits(direction):\n            gate('CX',control,q)\n    def product(a,b,direction):\n        if not a or not b or not direction:\n            return\n        if a == b:\n            for i in set_bits(a):\n                linear_output(basis[i],direction)\n            return\n        p = (a & -a).bit_length()-1\n        first = list(set_bits(a ^ (1 << p)))\n        for i in first:\n            gate('CX',basis[i],basis[p])\n        if b >> p & 1:\n            b ^= a ^ (1 << p)\n        choices = b & ~(1 << p)\n        assert choices\n        q = (choices & -choices).bit_length()-1\n        second = list(set_bits(b ^ (1 << q)))\n        for i in second:\n            gate('CX',basis[i],basis[q])\n        pivot,*others = target_bits(direction)\n        for out in others:\n            gate('CX',pivot,out)\n        gate('CCX',basis[p],basis[q],pivot)\n        for out in others:\n            gate('CX',pivot,out)\n        for i in reversed(second):\n            gate('CX',basis[i],basis[q])\n        for i in reversed(first):\n            gate('CX',basis[i],basis[p])\n    high_only = [sum(int(ac[h,0,j]) << j for j in range(O)) for h in range(A)]\n    low_only = [sum(int(ac[0,l,j]) << j for j in range(O)) for l in range(B)]\n    low_only[0] = 0\n    def correction(destination,left,right,direction):\n        for a in set_bits(left):\n            for b in set_bits(right):\n                index = (a+1)|(b+1)\n                if index < len(destination):\n                    destination[index] ^= direction\n    for term in terms:\n        pairs = term['factors']\n        direction = term['target_direction']\n        for i in range(0,len(pairs),2):\n            h1,l1 = pairs[i]\n            if i+1 == len(pairs):\n                product(h1,l1 << H,direction)\n            else:\n                h2,l2 = pairs[i+1]\n                # (h1+l2)(h2+l1) + h1*h2 + l1*l2\n                # equals h1*l1 + h2*l2 over GF(2).\n                product(h1 ^ (l2 << H),h2 ^ (l1 << H),direction)\n                correction(high_only,h1,h2,direction)\n                correction(low_only,l1,l2,direction)\n    for q in target_bits(high_only[0]):\n        gate('X',q)\n    for mask in range(1,A):\n        linear_output(high[mask],high_only[mask])\n    for mask in range(1,B):\n        linear_output(low[mask],low_only[mask])\n    for triple in reversed(cache):\n        gate('CCX',*triple)\n    circuit = builder.finish_circuit()\n    native = int(circuit.operation_statistics().num_toffoli_like)\n    assert native == sum(t[0]=='CCX' for t in trace)\n    return circuit,trace,{'num_qubits':int(circuit.num_qubits),\n        'toffoli_count':native,'clean_cached_functions':len(cache),\n        'coherent_cache_compute_and_cleanup_toffolis':2*len(cache),\n        'gates':len(trace)}\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument('--angle-words',type=Path,required=True)\n    p.add_argument('--tree',type=Path,required=True)\n    p.add_argument('--source-factorization',type=Path,required=True)\n    p.add_argument('--input-brot',type=int,default=12)\n    p.add_argument('--brot',type=int,default=8)\n    p.add_argument('--rows',type=int,default=700)\n    p.add_argument('--batch-size',type=int,default=5)\n    p.add_argument('--lowbits',type=int,default=4)\n    p.add_argument('--k',type=int,default=0)\n    p.add_argument('--max-kernels',type=int,default=32767)\n    p.add_argument('--out',type=Path,required=True)\n    p.add_argument('--verifier-dir',type=Path,default=Path('/tmp/chem_verifiers/dfthcsos'))\n    a = p.parse_args()\n    start = time.time()\n    if a.out.exists():\n        raise FileExistsError('refusing to overwrite previous results')\n    original = np.load(a.angle_words,allow_pickle=False)[:a.rows].copy()\n    edges = np.load(a.tree,allow_pickle=False)\n    if a.input_brot < a.brot:\n        raise ValueError('input precision must be at least output precision')\n    shift = a.input_brot-a.brot\n    rounded = ((original+(1 << (shift-1))) >> shift) % (1 << a.brot) if shift else original\n    words,signs = canonical_gauge(rounded,edges,a.brot)\n    gauge_error = float(np.max(np.abs(decode(words,edges,a.brot)-\n                                     signs[:,None]*decode(rounded,edges,a.brot))))\n    assert gauge_error < 1e-12\n    delta = delta_batch(words,a.batch_size,a.k)\n    ac,terms,planning = plan(delta,a.brot,a.lowbits,a.max_kernels)\n    circuit,trace,report = build(delta,a.brot,a.lowbits,ac,terms)\n    report['exact_symbolic_oracle'] = symbolic_check(trace,delta,a.brot,circuit.num_qubits)\n    report['native_oracle'] = native_check(circuit,delta,a.brot,a.verifier_dir)\n    a.out.mkdir(parents=True)\n    np.save(a.out/'fixture_angle_words.npy',words)\n    np.save(a.out/'batch_delta.npy',delta)\n    np.save(a.out/'anf_coefficients.npy',ac)\n    (a.out/'certificate.json').write_text(json.dumps({'plan':planning,'terms':terms},indent=2)+'\\n')\n    cp = a.out/f'angle_k_{a.k}.kmb'\n    circuit.to_file(cp,format='kmb')\n    (a.out/f'angle_k_{a.k}.kmx').write_text(str(circuit))\n    report.update({'scope':'ORGANIZER_SUBSET_CIRCUIT_FIXTURE_ONLY',\n        'chemistry_acceptance_claimed':False,'full_family_compiled':False,\n        'D':len(words),'brot':a.brot,'batch_size':a.batch_size,'k':a.k,\n        'lowbits':a.lowbits,'planning':planning,'gauge_max_coordinate_error':gauge_error,\n        'source_factorization_sha256':sha(a.source_factorization),\n        'source_words_sha256':sha(a.angle_words),'source_tree_sha256':sha(a.tree),\n        'fixture_words_sha256':sha(a.out/'fixture_angle_words.npy'),\n        'compiler_sha256':sha(__file__),'proof_helper_sha256':sha(Path(__file__).with_name('borrowed_prefix.py')),\n        'native_sha256':sha(cp),'seconds':time.time()-start})\n    (a.out/'report.json').write_text(json.dumps(report,indent=2)+'\\n')\n    print(json.dumps({k:v for k,v in report.items() if k!='planning'},indent=2))\n    print(json.dumps({k:v for k,v in planning.items() if k not in ['output_basis_certificates','original_matrix_ranks']},indent=2))\n\n\nif __name__ == '__main__':\n    main()\n","borrowed_prefix.py":"#!/usr/bin/env python3\n\"\"\"Organizer-only coherent prefix lookup with restored dirty output helpers.\n\nThe source table is required explicitly. This file imports no fitted data or\nprevious decomposition certificate. It only emits X, CX and CCX gates.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef mobius(table):\n    result = np.asarray(table, dtype=np.int64).copy()\n    ids = np.arange(len(result))\n    for bit in range((len(result)-1).bit_length()):\n        rows = ids[(ids & (1 << bit)) != 0]\n        result[rows] ^= result[rows ^ (1 << bit)]\n    return result\n\n\ndef delta_batch(words, size, k):\n    def part(index):\n        result = np.zeros((len(words), size), dtype=np.int64)\n        if index >= 0:\n            start = index * size\n            width = max(0, min(size, words.shape[1] - start))\n            result[:, :width] = words[:, start:start+width]\n        return result\n    return part(k) ^ part(k-1)\n\n\ndef build(delta, brot, clean_cap):\n    import kickmix as km\n    count, size = delta.shape\n    n = (count-1).bit_length()\n    coefficients = mobius(delta)\n    assert np.array_equal(mobius(coefficients), delta)\n    used = set(map(int, np.flatnonzero(np.any(coefficients != 0, axis=1))))\n    for mask in tuple(used):\n        while mask:\n            mask ^= 1 << (mask.bit_length()-1)\n            if mask:\n                used.add(mask)\n    children = {mask: [] for mask in used if mask}\n    for mask in sorted(used):\n        if mask.bit_count() > 1:\n            children[mask ^ (1 << (mask.bit_length()-1))].append(mask)\n\n    builder = km.CircuitBuilder()\n    qubits = list(builder.create_quantum_register(n, 'selection'))\n    for j in range(size):\n        qubits += list(builder.create_quantum_register(brot, f'word_{j}'))\n    target_ids = list(range(n, n + brot*size))\n    scratch_start = len(qubits)\n    qubits += list(builder.alloc_qubits(clean_cap))\n    if len(qubits) > 128:\n        raise ValueError(f'layout requires {len(qubits)} qubits')\n    trace = []\n    fallback = []\n\n    def gate(name, *ids):\n        trace.append((name, *ids))\n        getattr(builder, name.lower())(*[qubits[x] for x in ids])\n\n    def targets(mask):\n        return [n + j*brot + bit for j in range(size)\n                for bit in range(brot) if int(coefficients[mask,j]) >> bit & 1]\n\n    def multi_x(controls, target):\n        k = len(controls)\n        if k <= 2:\n            gate(('X', 'CX', 'CCX')[k], *controls, target)\n            return\n        dirty = [q for q in target_ids if q != target][:k-2]\n        if len(dirty) != k-2:\n            raise ValueError('not enough distinct borrowed output bits')\n        # B = U CCX(c0,c1,d0) U^-1 toggles the last dirty bit by\n        # product(c0,...,c[k-2]). T B T B then toggles only the target.\n        # There are 4*k-8 CCX gates, and every dirty bit is restored.\n        def bridge():\n            for i in reversed(range(1,k-2)):\n                gate('CCX', dirty[i-1], controls[i+1], dirty[i])\n            gate('CCX', controls[0], controls[1], dirty[0])\n            for i in range(1,k-2):\n                gate('CCX', dirty[i-1], controls[i+1], dirty[i])\n        gate('CCX', dirty[-1], controls[-1], target)\n        bridge()\n        gate('CCX', dirty[-1], controls[-1], target)\n        bridge()\n\n    def fan_product(controls, outputs):\n        if not outputs:\n            return\n        pivot, *others = outputs\n        for q in others:\n            gate('CX', pivot, q)\n        multi_x(controls, pivot)\n        for q in others:\n            gate('CX', pivot, q)\n\n    def direct_subtree(mask, ancestor, control):\n        outputs = targets(mask)\n        if outputs:\n            rest = mask ^ ancestor\n            controls = [control] + [bit for bit in range(n) if rest >> bit & 1]\n            fan_product(controls, outputs)\n            fallback.append({'mask': mask, 'controls':len(controls),\n                             'ccx': 1 if len(controls)==2 else 4*len(controls)-8})\n        for child in children[mask]:\n            direct_subtree(child, ancestor, control)\n\n    def visit(mask, control):\n        for q in targets(mask):\n            gate('CX', control, q)\n        for child in children[mask]:\n            new_bit = child.bit_length()-1\n            if not children[child]:\n                fan_product([control, new_bit], targets(child))\n            elif child.bit_count()-1 <= clean_cap:\n                temporary = scratch_start + child.bit_count()-2\n                gate('CCX', control, new_bit, temporary)\n                visit(child, temporary)\n                gate('CCX', control, new_bit, temporary)\n            else:\n                direct_subtree(child, mask, control)\n\n    for q in targets(0):\n        gate('X', q)\n    for bit in range(n):\n        if 1 << bit in used:\n            visit(1 << bit, bit)\n    circuit = builder.finish_circuit()\n    num_ccx = sum(t[0] == 'CCX' for t in trace)\n    assert int(circuit.operation_statistics().num_toffoli_like) == num_ccx\n    return circuit, trace, {'num_qubits':int(circuit.num_qubits),\n        'toffoli_count':num_ccx, 'clean_scratch':clean_cap,\n        'nonlinear_anf_terms':sum(m.bit_count()>1 for m in used),\n        'fallback_products':fallback, 'gates':len(trace)}\n\n\ndef symbolic_check(trace, delta, brot, num_qubits):\n    \"\"\"Exact address-packed affine labels cover every arbitrary target state.\"\"\"\n    count, size = delta.shape\n    n = (count-1).bit_length()\n    all_addresses = (1 << count)-1\n    def packed(bits):\n        return int.from_bytes(np.packbits(np.asarray(bits,dtype=np.uint8),\n                                         bitorder='little').tobytes(), 'little')\n    addresses = np.arange(count, dtype=np.int64)\n    states = [{-1:packed(addresses >> bit & 1)} for bit in range(n)]\n    states += [{i:all_addresses} for i in range(size*brot)]\n    states += [{} for _ in range(num_qubits-len(states))]\n    def add(target, source):\n        for label, bits in source.items():\n            value = states[target].get(label,0) ^ bits\n            if value:\n                states[target][label] = value\n            else:\n                states[target].pop(label,None)\n    for instruction in trace:\n        name, *q = instruction\n        if name == 'X':\n            add(q[0], {-1:all_addresses})\n        elif name == 'CX':\n            add(q[1], states[q[0]])\n        elif name == 'CCX':\n            left, right = states[q[0]], states[q[1]]\n            if any(k >= 0 for k in left):\n                left, right = right, left\n            if any(k >= 0 for k in left):\n                raise AssertionError('unexpected target-quadratic intermediate')\n            control = left.get(-1,0)\n            add(q[2], {key:value & control for key,value in right.items()})\n        else:\n            raise AssertionError(f'non-classical gate {name}')\n    for bit in range(n):\n        assert states[bit] == {-1:packed(addresses >> bit & 1)}\n    for j in range(size):\n        for bit in range(brot):\n            expected = {j*brot+bit:all_addresses}\n            value = packed(delta[:,j] >> bit & 1)\n            if value:\n                expected[-1] = value\n            assert states[n+j*brot+bit] == expected\n    assert all(not s for s in states[n+size*brot:])\n    return {'status':'PASS', 'all_valid_addresses':count,\n            'arbitrary_target_bits':size*brot,\n            'target_domain':f'all 2**{size*brot} assignments',\n            'all_clean_scratch_restored':True,\n            'all_selection_bits_restored':True,\n            'zero_phase':'Only unconditional X/CX/CCX permutation gates'}\n\n\ndef native_check(circuit, delta, brot, verifier, repeats=2):\n    import kickmix as km\n    path = Path(verifier)/'scoreinfra/circuits.py'\n    spec = importlib.util.spec_from_file_location('_organizer_lookup_contract',path)\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[spec.name] = module\n    spec.loader.exec_module(module)\n    layout = module.register_layout([circuit])\n    count = len(delta)\n    addresses = np.tile(np.arange(count,dtype=np.int64),repeats)\n    rng = np.random.default_rng(902113)\n    initial = rng.integers(0,1 << brot,size=(len(addresses),delta.shape[1]))\n    bits = ((addresses[None,:] >> np.arange(len(layout.selection))[:,None]) & 1).astype(bool)\n    sim = km.Simulator(batch_size=len(addresses),debug_print=False,seed=902113,\n                       counting_operations=True)\n    sim.use_same_registers_as(circuit)\n    sim.write_across_shots(layout.selection,bits)\n    for j,reg in enumerate(layout.words):\n        sim.write_across_shots(reg,((initial[:,j][None,:] >> np.arange(brot)[:,None]) & 1).astype(bool))\n    sim.do(circuit)\n    module._check_state(sim,circuit,layout,bits,'organizer borrowed-prefix probe')\n    powers = np.int64(1) << np.arange(brot)\n    actual = np.stack([powers @ sim.read_across_shots(reg).astype(np.int64)\n                       for reg in layout.words],axis=1)\n    assert np.array_equal(actual, initial ^ delta[addresses])\n    assert sim.operation_statistics().num_toffoli_like == len(addresses)*circuit.operation_statistics().num_toffoli_like\n    return {'status':'PASS','actual_native_arbitrary_target_shots':len(addresses),\n            'scratch_phase_selection_checked':True}\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument('--angle-words',type=Path,required=True)\n    p.add_argument('--source-factorization',type=Path,required=True)\n    p.add_argument('--brot',type=int,default=12)\n    p.add_argument('--batch-size',type=int,default=9)\n    p.add_argument('--k',type=int,default=0)\n    p.add_argument('--clean-cap',type=int,default=8)\n    p.add_argument('--out',type=Path,required=True)\n    p.add_argument('--verifier-dir',type=Path,default=Path('/tmp/chem_verifiers/dfthcsos'))\n    a = p.parse_args()\n    start = time.time()\n    if a.out.exists():\n        raise FileExistsError('refusing to overwrite a probe directory')\n    words = np.load(a.angle_words,allow_pickle=False)\n    if isinstance(words,np.lib.npyio.NpzFile):\n        words = words['angle_words'].copy()\n    if np.any(words < 0) or np.any(words >= 1 << a.brot):\n        raise ValueError('word outside the selected precision')\n    delta = delta_batch(words,a.batch_size,a.k)\n    circuit, trace, report = build(delta,a.brot,a.clean_cap)\n    report['symbolic_oracle'] = symbolic_check(trace,delta,a.brot,circuit.num_qubits)\n    report['native_oracle'] = native_check(circuit,delta,a.brot,a.verifier_dir)\n    a.out.mkdir(parents=True)\n    cp = a.out/f'angle_k_{a.k}.kmb'\n    circuit.to_file(cp,format='kmb')\n    (a.out/f'angle_k_{a.k}.kmx').write_text(str(circuit))\n    np.save(a.out/'batch_delta.npy',delta)\n    report.update({'scope':'ORGANIZER_ONLY_EXACT_CIRCUIT_COMPONENT',\n        'full_family_compiled':False,'chemistry_acceptance_claimed':False,\n        'brot':a.brot,'batch_size':a.batch_size,'k':a.k,\n        'source_factorization':str(a.source_factorization.resolve()),\n        'source_factorization_sha256':sha(a.source_factorization),\n        'angle_words':str(a.angle_words.resolve()),'angle_words_sha256':sha(a.angle_words),\n        'compiler_sha256':sha(__file__),'native_sha256':sha(cp),\n        'seconds':time.time()-start})\n    (a.out/'report.json').write_text(json.dumps(report,indent=2)+'\\n')\n    print(json.dumps(report,indent=2))\n\n\nif __name__ == '__main__':\n    main()\n","kernel_planes.py":"#!/usr/bin/env python3\n\"\"\"Exact multi-output product sharing on a single organizer-only fixture.\n\nThe search chooses a small subspace of the coefficient tensor's exact kernel.\nRank-two product templates are admitted only when BOTH of their coefficient\ncolumns annihilate those relations. A greedy paired basis search is accepted\nonly after exact reconstruction of every tensor coefficient. No randomized\nrank test or chemistry approximation is used as an acceptance criterion.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nimport random\nimport time\n\nimport numpy as np\n\nimport rank_parity as core\n\n\ndef packed(row):\n    return sum(int(value) << j for j,value in enumerate(row))\n\n\ndef tensor_column(direction,lowform,m):\n    return sum(direction << (m*j) for j in core.set_bits(lowform))\n\n\ndef low_planes():\n    planes = set()\n    def add(a,b):\n        first,second,_ = sorted((a,b,a^b))\n        planes.add((first,second))\n    for shift in (0,3):\n        for a in range(1,8):\n            for b in range(a+1,8):\n                add(a << shift,b << shift)\n    for a in range(1,8):\n        for b in range(1,8):\n            add(a,b << 3)\n        add(a,1 << 6)\n        add(a << 3,1 << 6)\n    return sorted(planes)\n\n\ndef reduce_row(value,pivots):\n    while value:\n        p = value.bit_length()-1\n        if p not in pivots:\n            break\n        value ^= pivots[p]\n    return value\n\n\ndef append_row(value,pivots):\n    value = reduce_row(value,pivots)\n    if not value:\n        return False\n    pivots[value.bit_length()-1] = value\n    return True\n\n\ndef select_relations(rows,width,dimension,rng):\n    kernel = core.nullspace(rows,width)\n    if len(kernel) < dimension:\n        raise ValueError('requested kernel dimension exceeds the actual nullity')\n    chosen = []\n    span = {}\n    while len(chosen) < dimension:\n        mask = rng.getrandbits(len(kernel))\n        value = 0\n        for j in core.set_bits(mask):\n            value ^= kernel[j]\n        if append_row(value,span):\n            assert all((value & row).bit_count()%2 == 0 for row in rows)\n            chosen.append(value)\n    return chosen,len(kernel)\n\n\ndef paired_basis(templates,target_rank,seed,trials):\n    rng = random.Random(seed)\n    best = 0\n    reports = []\n    for trial in range(trials):\n        order = list(range(len(templates)))\n        rng.shuffle(order)\n        span = {}\n        chosen = []\n        for i in order:\n            a,b,*_ = templates[i]\n            ar = reduce_row(a,span)\n            if not ar:\n                continue\n            p = ar.bit_length()-1\n            span[p] = ar\n            br = reduce_row(b,span)\n            if not br:\n                del span[p]\n                continue\n            span[br.bit_length()-1] = br\n            chosen.append(i)\n            if len(span) == target_rank:\n                reports.append({'trial':trial,'rank':len(span)})\n                return chosen,reports\n        best = max(best,len(span))\n        reports.append({'trial':trial,'rank':len(span)})\n    return None,reports\n\n\ndef plan(delta,brot,dimensions=(15,15),seed=0,pair_trials=16,samples=12):\n    core.require_redundant_sign_bit(delta,brot)\n    ac = core.coefficients(delta,brot,3)\n    tensor = ac[1:,1:,:]\n    H,L,m = tensor.shape\n    if L != 7:\n        raise ValueError('this bounded planner requires exactly three low address bits')\n    F = tensor.reshape(H,L*m)\n    rows = [packed(row) for row in F]\n    rng = random.Random(seed)\n    relations = []\n    nullities = []\n    for group,dimension in enumerate(dimensions):\n        block_rows = [packed(row) for row in F[:,3*m*group:3*m*(group+1)]]\n        chosen,nullity = select_relations(block_rows,3*m,dimension,rng)\n        relations += [value << (3*m*group) for value in chosen]\n        nullities.append(nullity)\n    assert all((row & relation).bit_count()%2 == 0 for row in rows for relation in relations)\n    mask = (1 << m)-1\n    chunks = [[(r >> (j*m)) & mask for j in range(L)] for r in relations]\n    templates = []\n    plane_dimensions = []\n    for ell1,ell2 in low_planes():\n        equations = []\n        for row in chunks:\n            for ell in (ell1,ell2):\n                value = 0\n                for j in core.set_bits(ell):\n                    value ^= row[j]\n                if value:\n                    equations.append(value)\n        V = core.nullspace(equations,m)\n        plane_dimensions.append(len(V))\n        directions = set(V)\n        for _ in range(samples):\n            choice = rng.getrandbits(len(V))\n            value = 0\n            for j in core.set_bits(choice):\n                value ^= V[j]\n            if value:\n                directions.add(value)\n        for direction in sorted(directions):\n            a = tensor_column(direction,ell1,m)\n            b = tensor_column(direction,ell2,m)\n            assert all((a&r).bit_count()%2 == 0 and (b&r).bit_count()%2 == 0 for r in relations)\n            templates.append((a,b,direction,ell1,ell2))\n    all_span = {}\n    for a,b,*_ in templates:\n        append_row(a,all_span)\n        append_row(b,all_span)\n    target_rank = L*m-sum(dimensions)\n    contains_target = all(not reduce_row(row,all_span) for row in rows)\n    report = {'H':H,'L':L,'O':m,'kernel_dimensions':list(dimensions),\n        'kernel_nullities':nullities,'kernel_relations':relations,\n        'num_templates':len(templates),'template_span_rank':len(all_span),\n        'desired_span_rank':target_rank,'contains_target':contains_target,\n        'plane_output_nullities':plane_dimensions,'seed':seed}\n    if len(all_span) != target_rank or not contains_target:\n        return None,report\n    indices,attempts = paired_basis(templates,target_rank,seed+1001,pair_trials)\n    report['paired_basis_attempts'] = attempts\n    if indices is None:\n        return None,report\n    chosen = [templates[i] for i in indices]\n    basis = core.BinaryBasis()\n    for a,b,*_ in chosen:\n        assert basis.add(a) and basis.add(b)\n    solutions = [basis.solve(row) for row in rows]\n    assert all(value is not None for value in solutions)\n    terms = []\n    reconstructed = [0]*H\n    for i,(a,b,direction,ell1,ell2) in enumerate(chosen):\n        h1 = sum(((value >> (2*i))&1) << h for h,value in enumerate(solutions))\n        h2 = sum(((value >> (2*i+1))&1) << h for h,value in enumerate(solutions))\n        terms.append({'target_direction':direction,'factors':[(h1,ell1),(h2,ell2)]})\n        for h in core.set_bits(h1):\n            reconstructed[h] ^= a\n        for h in core.set_bits(h2):\n            reconstructed[h] ^= b\n    assert reconstructed == rows\n    report.update({'selected_pairs':len(chosen),'exact_selected_binary_rank':len(basis.vectors),\n        'exact_tensor_reconstruction':True,'selected_template_indices':indices})\n    return (ac,terms),report\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument('--fixture-dir',type=Path,required=True)\n    p.add_argument('--dimensions',type=int,nargs=2,default=(15,15))\n    p.add_argument('--seed',type=int,default=0)\n    p.add_argument('--pair-trials',type=int,default=16)\n    p.add_argument('--samples',type=int,default=12)\n    p.add_argument('--out',type=Path,required=True)\n    p.add_argument('--verifier-dir',type=Path,default=Path('/tmp/chem_verifiers/dfthcsos'))\n    a = p.parse_args()\n    start = time.time()\n    if a.out.exists():\n        raise FileExistsError('refusing to overwrite an experiment')\n    source = json.loads((a.fixture_dir/'report.json').read_text())\n    if source['scope'] != 'ORGANIZER_SUBSET_CIRCUIT_FIXTURE_ONLY':\n        raise ValueError('this bounded experiment requires the organizer fixture')\n    delta = np.load(a.fixture_dir/'batch_delta.npy',allow_pickle=False)\n    result,planning = plan(delta,source['brot'],tuple(a.dimensions),a.seed,a.pair_trials,a.samples)\n    a.out.mkdir(parents=True)\n    if result is None:\n        report={'status':'NO_EXACT_PAIRED_PLAN','planning':planning,'seconds':time.time()-start}\n        (a.out/'report.json').write_text(json.dumps(report,indent=2)+'\\n')\n        print(json.dumps({k:v for k,v in report.items() if k!='planning'}))\n        print(json.dumps({k:v for k,v in planning.items() if k not in ['kernel_relations','plane_output_nullities']}))\n        return\n    ac,terms = result\n    circuit,trace,report = core.build(delta,source['brot'],3,ac,terms)\n    report['exact_symbolic_oracle'] = core.symbolic_check(trace,delta,source['brot'],circuit.num_qubits)\n    report['native_oracle'] = core.native_check(circuit,delta,source['brot'],a.verifier_dir)\n    cp = a.out/'angle_k_0.kmb'\n    circuit.to_file(cp,format='kmb')\n    (a.out/'angle_k_0.kmx').write_text(str(circuit))\n    (a.out/'certificate.json').write_text(json.dumps({'planning':planning,'terms':terms},indent=2)+'\\n')\n    report.update({'status':'PASS','scope':'ORGANIZER_SUBSET_CIRCUIT_FIXTURE_ONLY',\n        'chemistry_acceptance_claimed':False,'full_family_compiled':False,\n        'D':len(delta),'brot':source['brot'],'batch_size':source['batch_size'],\n        'planning':planning,'fixture_source_report_sha256':core.sha(a.fixture_dir/'report.json'),\n        'fixture_delta_sha256':core.sha(a.fixture_dir/'batch_delta.npy'),\n        'planner_sha256':core.sha(__file__),'emitter_sha256':core.sha(core.__file__),\n        'proof_helper_sha256':core.sha(Path(__file__).with_name('borrowed_prefix.py')),\n        'native_sha256':core.sha(cp),'seconds':time.time()-start})\n    (a.out/'report.json').write_text(json.dumps(report,indent=2)+'\\n')\n    print(json.dumps({k:v for k,v in report.items() if k!='planning'},indent=2))\n    print(json.dumps({k:v for k,v in planning.items() if k not in ['kernel_relations','plane_output_nullities']},indent=2))\n\n\nif __name__=='__main__':\n    main()\n","permuted_kernel.py":"\"\"\"Three-low-direction kernel cycles for exact J4 bilinear synthesis.\"\"\"\nfrom pathlib import Path\nimport sys,json,time,random,itertools\nimport numpy as np\nROOT=Path(__file__).resolve().parent\nsys.path.insert(0,str(ROOT))\nimport rank_parity as core\nimport kernel_planes as binary\n\ndef quotient_selection(templates,target,L,seed,trials=64):\n    W0={};coordinate=[];spaces=[{} for _ in range(L)]\n    for i,(a,b,v,ell1,ell2) in enumerate(templates):\n        if ell1.bit_count()==ell2.bit_count()==1:\n            left=ell1.bit_length()-1;right=ell2.bit_length()-1\n            coordinate.append((i,left,right,v))\n            binary.append_row(a,W0);binary.append_row(b,W0)\n            binary.append_row(v,spaces[left]);binary.append_row(v,spaces[right])\n    pivots=sorted(W0.items(),reverse=True)\n    def quotient(v):\n        for p,row in pivots:\n            if v>>p&1:v^=row\n        return v\n    mixed=[]\n    for i,(a,b,v,ell1,ell2) in enumerate(templates):\n        ar,br=quotient(a),quotient(b)\n        if ar and not br and ell2.bit_count()==1:mixed.append((i,ar,ell2.bit_length()-1,v))\n        elif br and not ar and ell1.bit_count()==1:mixed.append((i,br,ell1.bit_length()-1,v))\n    qrank=target-len(W0);dims=[len(x) for x in spaces];rng=random.Random(seed);attempts=[]\n    for trial in range(trials):\n        order=mixed.copy();rng.shuffle(order);qspan={};local=[{} for _ in range(L)];chosen=[]\n        for i,q,vertex,v in order:\n            qr=binary.reduce_row(q,qspan);vr=binary.reduce_row(v,local[vertex])\n            if qr and vr:\n                qspan[qr.bit_length()-1]=qr;local[vertex][vr.bit_length()-1]=vr;chosen.append(i)\n        if len(qspan)!=qrank:\n            attempts.append({'trial':trial,'quotient_rank':len(qspan),'required':qrank});continue\n        pool=coordinate.copy();rng.shuffle(pool)\n        while 2*len(chosen)<target:\n            deficit=[dims[j]-len(local[j]) for j in range(L)];best=None\n            for i,left,right,v in pool:\n                if not deficit[left] or not deficit[right]:continue\n                lr=binary.reduce_row(v,local[left]);rr=binary.reduce_row(v,local[right])\n                if not lr or not rr:continue\n                score=(deficit[left]+deficit[right],min(deficit[left],deficit[right]))\n                if best is None or score>best[0]:best=(score,i,left,right,lr,rr)\n            if best is None:break\n            _,i,left,right,lr,rr=best;local[left][lr.bit_length()-1]=lr;local[right][rr.bit_length()-1]=rr;chosen.append(i)\n        span={}\n        for i in chosen:\n            binary.append_row(templates[i][0],span);binary.append_row(templates[i][1],span)\n        attempts.append({'trial':trial,'quotient_rank':len(qspan),'rank':len(span),'pairs':len(chosen)})\n        if 2*len(chosen)==len(span)==target:return chosen,{'W0':len(W0),'qrank':qrank,'attempts':attempts}\n    return None,{'W0':len(W0),'qrank':qrank,'attempts':attempts}\n\ndef plan(delta,extra=15,seed=0,permutation=None):\n    permutation=list(range(15)) if permutation is None else list(permutation)\n    assert sorted(permutation)==list(range(15))\n    original_ac=core.coefficients(delta,8,4); ac=original_ac.copy(); ac[:,1:,:]=ac[:,1:,:][:,permutation,:]; tensor=ac[1:,1:,:];H,L,m=tensor.shape\n    assert (L,m)==(15,28)\n    rows=[binary.packed(row) for row in tensor.reshape(H,L*m)]\n    rng=random.Random(seed);relations=[];nullities=[]\n    chosen_extra=set(rng.sample(range(L),extra))\n    for first in range(L):\n        vs=[(first+i)%L for i in range(3)]\n        block=np.concatenate([tensor[:,v,:] for v in vs],axis=1)\n        selected,nullity=binary.select_relations([binary.packed(row) for row in block],3*m,3+(first in chosen_extra),rng)\n        nullities.append(nullity)\n        for rel in selected:\n            relations.append(sum(((rel>>(j*m))&((1<<m)-1))<<(v*m) for j,v in enumerate(vs)))\n    rb={}\n    for r in relations:binary.append_row(r,rb)\n    assert all(not ((row&r).bit_count()&1) for row in rows for r in relations)\n    chunks=[[(r>>(j*m))&((1<<m)-1) for j in range(L)] for r in relations]\n    planes=[(1<<i,1<<j) for i in range(L) for j in range(i+1,L)]\n    for step in [1,2]:\n        planes.extend(((1<<i)|(1<<((i+step)%L)),1<<k) for i in range(L) for k in range(L) if k not in [i,(i+step)%L])\n    templates=[];dims=[]\n    for a,b in planes:\n        eq=[]\n        for cs in chunks:\n            for ell in [a,b]:\n                value=0\n                for j in core.set_bits(ell):value^=cs[j]\n                if value:eq.append(value)\n        V=core.nullspace(eq,m);dims.append(len(V))\n        directions=set(V)\n        if len(V)<=6:\n            for code in range(1,1<<len(V)):\n                direction=0\n                for j in core.set_bits(code):direction^=V[j]\n                directions.add(direction)\n        for v in sorted(directions):\n            if v:\n                u=binary.tensor_column(v,a,m);w=binary.tensor_column(v,b,m)\n                templates.append((u,w,v,a,b))\n    span={}\n    for a,b,*_ in templates:binary.append_row(a,span);binary.append_row(b,span)\n    target=L*m-len(rb)\n    report={'extra':extra,'seed':seed,'nullities':nullities,'relations':len(rb),'target_rank':target,'span_rank':len(span),'templates':len(templates),'plane_dim_hist':{i:dims.count(i) for i in sorted(set(dims))},'contains_target':all(not binary.reduce_row(row,span) for row in rows),'T_lower_if_paired':84+(target+1)//2}\n    if len(span)!=target or not report['contains_target'] or target%2:return None,report\n    selected,attempts=quotient_selection(templates,target,L,seed+7001)\n    report['paired_attempts']=attempts\n    if selected is None:return None,report\n    basis=core.BinaryBasis()\n    for i in selected:\n        a,b,*_=templates[i];assert basis.add(a) and basis.add(b)\n    solutions=[basis.solve(row) for row in rows];assert all(s is not None for s in solutions)\n    terms=[];reconstructed=[0]*H\n    for j,i in enumerate(selected):\n        a,b,v,ell1,ell2=templates[i]\n        h1=sum(((s>>(2*j))&1)<<h for h,s in enumerate(solutions));h2=sum(((s>>(2*j+1))&1)<<h for h,s in enumerate(solutions))\n        terms.append({'target_direction':v,'factors':[(h1,ell1),(h2,ell2)]})\n        for h in core.set_bits(h1):reconstructed[h]^=a\n        for h in core.set_bits(h2):reconstructed[h]^=b\n    assert reconstructed==rows\n    report.update({'paired_products':len(selected),'T_HMR':84+len(selected),'Q':127,'exact_tensor':True})\n    \n    for term in terms:\n        term['factors']=[(h,sum(1<<permutation[j] for j in core.set_bits(l))) for h,l in term['factors']]\n    report['low_permutation']=permutation\n    return (original_ac,terms),report\n\n","hyper_native.py":"from pathlib import Path\nimport sys\nfrom rank_parity import *\ndef build_hmr(delta,brot,lowbits,ac,terms):\n    import kickmix as km\n    delta = require_redundant_sign_bit(delta,brot)\n    D,J = delta.shape\n    n = (D-1).bit_length()\n    A,B,O = ac.shape\n    H,L = A-1,B-1\n    builder = km.CircuitBuilder()\n    qubits = list(builder.create_quantum_register(n,'selection'))\n    for j in range(J):\n        qubits += list(builder.create_quantum_register(brot,f'word_{j}'))\n    trace = []\n    def gate(name,*ids):\n        trace.append((name,*ids))\n        getattr(builder,name.lower())(*[qubits[i] for i in ids])\n    cache = []\n    def cached_basis(size,offset):\n        result = {0:None}\n        for mask in range(1,size):\n            bit = mask.bit_length()-1\n            if mask.bit_count()==1:\n                result[mask] = offset+bit\n            else:\n                parent = mask ^ (1 << bit)\n                q = len(qubits)\n                qubits.extend(builder.alloc_qubits(1))\n                result[mask] = q\n                triple = (result[parent],offset+bit,q)\n                gate('CCX',*triple)\n                cache.append(triple)\n        return result\n    high = cached_basis(A,lowbits)\n    low = cached_basis(B,0)\n    if len(qubits)>128:\n        raise ValueError(f'cache layout needs {len(qubits)} qubits')\n    basis = [high[i] for i in range(1,A)]+[low[i] for i in range(1,B)]\n    groups = [[n+j*brot+bit] + ([n+j*brot+brot-1] if bit==brot-2 else [])\n              for j in range(J) for bit in range(brot-1)]\n    def target_bits(direction):\n        return [q for j in set_bits(direction) for q in groups[j]]\n    def linear_output(control,direction):\n        for q in target_bits(direction):\n            gate('CX',control,q)\n    def product(a,b,direction):\n        if not a or not b or not direction:\n            return\n        if a == b:\n            for i in set_bits(a):\n                linear_output(basis[i],direction)\n            return\n        p = (a & -a).bit_length()-1\n        first = list(set_bits(a ^ (1 << p)))\n        for i in first:\n            gate('CX',basis[i],basis[p])\n        if b >> p & 1:\n            b ^= a ^ (1 << p)\n        choices = b & ~(1 << p)\n        assert choices\n        q = (choices & -choices).bit_length()-1\n        second = list(set_bits(b ^ (1 << q)))\n        for i in second:\n            gate('CX',basis[i],basis[q])\n        pivot,*others = target_bits(direction)\n        for out in others:\n            gate('CX',pivot,out)\n        gate('CCX',basis[p],basis[q],pivot)\n        for out in others:\n            gate('CX',pivot,out)\n        for i in reversed(second):\n            gate('CX',basis[i],basis[q])\n        for i in reversed(first):\n            gate('CX',basis[i],basis[p])\n    high_only = [sum(int(ac[h,0,j]) << j for j in range(O)) for h in range(A)]\n    low_only = [sum(int(ac[0,l,j]) << j for j in range(O)) for l in range(B)]\n    low_only[0] = 0\n    def correction(destination,left,right,direction):\n        for a in set_bits(left):\n            for b in set_bits(right):\n                index = (a+1)|(b+1)\n                if index < len(destination):\n                    destination[index] ^= direction\n    for term in terms:\n        pairs = term['factors']\n        direction = term['target_direction']\n        for i in range(0,len(pairs),2):\n            h1,l1 = pairs[i]\n            if i+1 == len(pairs):\n                product(h1,l1 << H,direction)\n            else:\n                h2,l2 = pairs[i+1]\n                # (h1+l2)(h2+l1) + h1*h2 + l1*l2\n                # equals h1*l1 + h2*l2 over GF(2).\n                product(h1 ^ (l2 << H),h2 ^ (l1 << H),direction)\n                correction(high_only,h1,h2,direction)\n                correction(low_only,l1,l2,direction)\n    for q in target_bits(high_only[0]):\n        gate('X',q)\n    for mask in range(1,A):\n        linear_output(high[mask],high_only[mask])\n    for mask in range(1,B):\n        linear_output(low[mask],low_only[mask])\n    for a,b,q in reversed(cache):\n        builder.del_and(qubits[a],qubits[b],qubits[q])\n    circuit = builder.finish_circuit()\n    native = int(circuit.operation_statistics().num_toffoli_like)\n    assert native == sum(t[0]=='CCX' for t in trace)\n    return circuit,trace,{'num_qubits':int(circuit.num_qubits),\n        'toffoli_count':native,'clean_cached_functions':len(cache),\n        'cache_compute_toffolis':len(cache), 'cache_HMR_cleanup':len(cache),\n        'gates':len(trace)}\n\n","native_checks.py":"import numpy as np\nimport kickmix as km\n\ndef check(circuit, delta, *, brot=11, repeats=4, seed=127813):\n    \"\"\"Check every valid address with arbitrary targets, phase and scratch.\"\"\"\n    count, word_count = delta.shape\n    n_sel = (count - 1).bit_length()\n    addresses = np.tile(np.arange(count), repeats)\n    rng = np.random.default_rng(seed)\n    initial = rng.integers(0, 1 << brot, size=(len(addresses), word_count))\n    if repeats >= 2:\n        initial[:count] = 0\n        initial[count:2*count] = (1 << brot) - 1\n    sim = km.Simulator(batch_size=len(addresses), seed=seed, counting_operations=True)\n    sim.use_same_registers_as(circuit)\n    sim.write_across_shots(\"selection\", ((addresses[None, :] >> np.arange(n_sel)[:, None]) & 1).astype(bool))\n    for j in range(word_count):\n        sim.write_across_shots(f\"word_{j}\", ((initial[:, j][None, :] >> np.arange(brot)[:, None]) & 1).astype(bool))\n    powers = np.int64(1) << np.arange(brot)\n    for application in range(2):\n        sim.do(circuit)\n        got = np.stack([powers @ sim.read_across_shots(f\"word_{j}\").astype(np.int64)\n                        for j in range(word_count)], axis=1)\n        expected = initial ^ delta[addresses] if application == 0 else initial\n        assert np.array_equal(got, expected), \"incorrect arbitrary-target XOR\"\n        assert not np.any(sim.read_phase_flipped_across_shots()), \"incorrect phase\"\n        decoded = (np.int64(1) << np.arange(n_sel)) @ sim.read_across_shots(\"selection\").astype(np.int64)\n        assert np.array_equal(decoded, addresses), \"selection changed\"\n        total = sim.read_across_shots(km.array([km.q(i) for i in range(circuit.num_qubits)]))\n        assert not np.any(total[n_sel + brot * word_count:]), \"scratch not returned to zero\"\n    return float(sim.operation_statistics().num_toffoli_like) / len(addresses) / 2\n","emit_j5.py":"# Derived from Ahmed Berat Ozer v8 emit_family.py (Apache 2.0).\n# New experiment: use five 8-bit outputs in freed QROM workspace;\n# select five relations per low-address block so the paired dimension is even.\n\"\"\"Derived J=5 / 8-bit ANGLE family for an arbitrary row count with mrmeows' exact paired-product lookup\n(Apache-2.0, source/chemistry_6p69/reproduction), generalized: the number of kernel relations per three-vertex\nblock is a search parameter, so tables with fewer high-address rows (fewer vectors) use their larger kernel.\n\nUsage: python emit_family.py <words .npy or submission .npz> <out dir> [jobs]\nWrites angle_k_*.kmb and plans.json; every circuit is natively checked on all addresses (64 targets each).\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport random\nimport sys\nimport time\nfrom concurrent.futures import ProcessPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\n\nREP=Path(__file__).resolve().parent\nsys.path.insert(0,str(REP))\nimport rank_parity as core  # noqa: E402\nimport kernel_planes as binary  # noqa: E402\nfrom permuted_kernel import quotient_selection  # noqa: E402\n\n\ndef plan_general(delta, rel_base=3, extra=15, seed=0, permutation=None):\n    permutation = list(range(15)) if permutation is None else list(permutation)\n    original_ac = core.coefficients(delta, 8, 4)\n    ac = original_ac.copy()\n    ac[:, 1:, :] = ac[:, 1:, :][:, permutation, :]\n    tensor = ac[1:, 1:, :]\n    H, L, m = tensor.shape\n    assert L == 15\n    rows = [binary.packed(row) for row in tensor.reshape(H, L * m)]\n    rng = random.Random(seed)\n    relations = []\n    chosen_extra = set(rng.sample(range(L), min(extra, L)))\n    for first in range(L):\n        vs = [(first + i) % L for i in range(3)]\n        block = np.concatenate([tensor[:, v, :] for v in vs], axis=1)\n        want = rel_base + (first in chosen_extra)\n        kern = core.nullspace([binary.packed(row) for row in block], 3 * m)\n        want = min(want, len(kern))\n        selected, _ = binary.select_relations([binary.packed(row) for row in block], 3 * m, want, rng)\n        for rel in selected:\n            relations.append(sum(((rel >> (j * m)) & ((1 << m) - 1)) << (v * m) for j, v in enumerate(vs)))\n    rb = {}\n    for r in relations:\n        binary.append_row(r, rb)\n    chunks = [[(r >> (j * m)) & ((1 << m) - 1) for j in range(L)] for r in relations]\n    planes = [(1 << i, 1 << j) for i in range(L) for j in range(i + 1, L)]\n    for step in [1, 2]:\n        planes.extend(((1 << i) | (1 << ((i + step) % L)), 1 << k) for i in range(L) for k in range(L)\n                      if k not in [i, (i + step) % L])\n    templates = []\n    for a, b in planes:\n        eq = []\n        for cs in chunks:\n            for ell in [a, b]:\n                value = 0\n                for j in core.set_bits(ell):\n                    value ^= cs[j]\n                if value:\n                    eq.append(value)\n        V = core.nullspace(eq, m)\n        directions = set(V)\n        if len(V) <= 6:\n            for code in range(1, 1 << len(V)):\n                direction = 0\n                for j in core.set_bits(code):\n                    direction ^= V[j]\n                directions.add(direction)\n        for v in sorted(directions):\n            if v:\n                templates.append((binary.tensor_column(v, a, m), binary.tensor_column(v, b, m), v, a, b))\n    span = {}\n    for a, b, *_ in templates:\n        binary.append_row(a, span)\n        binary.append_row(b, span)\n    target = L * m - len(rb)\n    contains = all(not binary.reduce_row(row, span) for row in rows)\n    if len(span) != target or not contains or target % 2:\n        return None, {'target': target, 'span': len(span), 'contains': contains}\n    selected, _ = quotient_selection(templates, target, L, seed + 7001)\n    if selected is None:\n        return None, {'target': target, 'paired': None}\n    basis = core.BinaryBasis()\n    for i in selected:\n        a, b, *_ = templates[i]\n        assert basis.add(a) and basis.add(b)\n    solutions = [basis.solve(row) for row in rows]\n    assert all(s is not None for s in solutions)\n    terms = []\n    reconstructed = [0] * H\n    for j, i in enumerate(selected):\n        a, b, v, ell1, ell2 = templates[i]\n        h1 = sum(((s >> (2 * j)) & 1) << h for h, s in enumerate(solutions))\n        h2 = sum(((s >> (2 * j + 1)) & 1) << h for h, s in enumerate(solutions))\n        terms.append({'target_direction': v, 'factors': [(h1, ell1), (h2, ell2)]})\n        for h in core.set_bits(h1):\n            reconstructed[h] ^= a\n        for h in core.set_bits(h2):\n            reconstructed[h] ^= b\n    assert reconstructed == rows\n    for term in terms:\n        term['factors'] = [(h, sum(1 << permutation[j] for j in core.set_bits(l))) for h, l in term['factors']]\n    return (original_ac, terms), {'target': target, 'paired': len(selected), 'relations': len(rb),\n                                  'rel_base': rel_base, 'extra': extra, 'seed': seed}\n\n\ndef best_plan(delta, budget_seconds=600, rel_bases=(4,)):\n    \"\"\"mrmeows' accepted plans all use 3 relations per block plus one extra on every block (60 in all),\n    with a searched seed and low-monomial permutation; search the same space until a plan exists.\"\"\"\n    t0 = time.time()\n    rng = random.Random(12345)\n    attempt = 0\n    while time.time() - t0 < budget_seconds:\n        perm = list(range(15))\n        if attempt:\n            rng.shuffle(perm)\n        for rel_base in rel_bases:\n            for seed in range(8):\n                try:\n                    res, rep = plan_general(delta, rel_base, 15, seed, perm)\n                except (ValueError, AssertionError):\n                    continue\n                if res is not None:\n                    rep['permutation'] = perm\n                    return res, rep\n        attempt += 1\n    return None\n\n\ndef build_one(args):\n    k, words, out = args\n    import kickmix as km\n    from hyper_native import build_hmr\n    from native_checks import check\n    delta = core.delta_batch(words, 5, k)\n    res = best_plan(delta)\n    if res is None:\n        return {'k': k, 'error': 'no plan'}\n    (ac, terms), rep = res\n    circuit, _, resources = build_hmr(delta, 8, 4, ac, terms)\n    path = Path(out) / f'angle_k_{k}.kmb'\n    circuit.to_file(path, format='kmb')\n    circuit = km.Circuit.from_file(path, format='kmb')\n    native = check(circuit, delta, brot=8, repeats=64, seed=150007 + k)\n    assert native == resources['toffoli_count']\n    return {'k': k, 'T': native, 'Q': int(circuit.num_qubits), **rep}\n\n\ndef main():\n    src, out = Path(sys.argv[1]), Path(sys.argv[2])\n    jobs = int(sys.argv[3]) if len(sys.argv) > 3 else 4\n    words = np.load(src)['angle_words'] if src.suffix == '.npz' else np.load(src)\n    words = np.asarray(words, dtype=np.int64)\n    out.mkdir(parents=True, exist_ok=True)\n    K = (words.shape[1] + 4) // 5\n    with ProcessPoolExecutor(max_workers=jobs) as pool:\n        reports = list(pool.map(build_one, [(k, words, str(out)) for k in range(K)]))\n    total = sum(r.get('T', 0) for r in reports)\n    (out / 'plans.json').write_text(json.dumps({'rows': int(words.shape[0]), 'per_pass': total, 'T_both_passes': 2 * total,\n                                                'circuits': reports}, indent=1))\n    print(json.dumps({'rows': int(words.shape[0]), 'per_pass': total, 'T_both': 2 * total,\n                      'per_circuit': [r.get('T') for r in reports], 'paired': [r.get('paired') for r in reports]}))\n\n\nif __name__ == '__main__':\n    main()\n","LICENSE":"\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      \"License\" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      \"Licensor\" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      \"Legal Entity\" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      \"control\" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      \"You\" (or \"Your\") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      \"Source\" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      \"Object\" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      \"Work\" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      \"Derivative Works\" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      \"Contribution\" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, \"submitted\"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as \"Not a Contribution.\"\n\n      \"Contributor\" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a \"NOTICE\" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an \"AS IS\" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets \"[]\"\n      replaced with your own identifying information. (Don't include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same \"printed page\" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the \"License\");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an \"AS IS\" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n","NOTICE":"Derived J5 compiler: Vladislav Lavrentev, 2026.\nBase generic kernel-relation emitter: Ahmed Berat Ozer, public v8 components, Apache 2.0.\nPaired-product kernel, prefix cache and HMR emitter: mrmeows, Apache 2.0, with credited predecessors.\n"}''')
def sha(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
DATA=next(p.parent for p in INPUT.rglob('dqi.kmb.xz') if (p.parent/'dfthcsos/submission.npz').is_file() and sha(p.parent/'dfthcsos/submission.npz')==PIN['members']['dfthcsos/submission.npz'])
assert sha(DATA/'dqi.kmb.xz')==PIN['dqi_xz_sha256']
with lzma.open(DATA/'dqi.kmb.xz') as source,(FINAL/'dqi.kmb').open('wb') as target:shutil.copyfileobj(source,target,8<<20)
assert sha(FINAL/'dqi.kmb')==PIN['members']['dqi.kmb']
ARCHIVE=next(p for p in INPUT.rglob('submission.zip') if sha(p)==PIN['invsqrt_source_archive'])
with zipfile.ZipFile(ARCHIVE) as z:
    with z.open('invsqrt.kmb') as source,(FINAL/'invsqrt.kmb').open('wb') as target:shutil.copyfileobj(source,target,8<<20)
assert sha(FINAL/'invsqrt.kmb')==PIN['members']['invsqrt.kmb']
shutil.copy2(DATA/'dfthcsos/submission.npz',FINAL/'dfthcsos/submission.npz')
SOURCE_DIR=WORK/'compiler_src';SOURCE_DIR.mkdir(exist_ok=True)
for name,text in COMPILER_SOURCE.items():(SOURCE_DIR/name).write_text(text)
(SOURCE_DIR/'plans.json').write_text(json.dumps(PLANS,indent=2))
sys.path.insert(0,str(SOURCE_DIR))
import emit_j5 as E
from hyper_native import build_hmr
words=np.asarray(np.load(FINAL/'dfthcsos/submission.npz',allow_pickle=False)['angle_words'],dtype=np.int64)
assert words.shape==(1096,75) and len(PLANS)==15
t0=time.monotonic();BUILD=[]
for plan in PLANS:
    k=plan['k'];delta=E.core.delta_batch(words,5,k)
    result,certificate=E.plan_general(delta,plan['rel_base'],plan['extra'],plan['seed'],plan['permutation'])
    assert result is not None,('Certificate regeneration failed',k)
    ac,terms=result
    circuit,_,resources=build_hmr(delta,8,4,ac,terms)
    assert circuit.num_qubits==123 and resources['toffoli_count']==297
    path=FINAL/f'dfthcsos/angle_k_{k}.kmb';circuit.to_file(path,format='kmb')
    assert sha(path)==PIN['members'][f'dfthcsos/angle_k_{k}.kmb'],('Compiled bytes differ',k,sha(path))
    BUILD.append({'k':k,'Toffolis':297,'Qubits':123,'paired products':certificate['paired'],'kernel relations':certificate['relations']})
    del circuit
MEMBERS={p.relative_to(FINAL).as_posix():sha(p) for p in sorted(FINAL.rglob('*')) if p.is_file()}
assert MEMBERS==PIN['members'] and len(MEMBERS)==18
def environment_fingerprint():
    files=[]
    for root in [COMP/'kickmix',COMP/'verifiers']:
        files.extend(p for p in root.rglob('*') if p.is_file() and p.suffix in {'.py','.so','.npz','.fcidump'})
    assert any(p.suffix=='.so' for p in files)
    return {p.relative_to(COMP).as_posix():sha(p) for p in sorted(set(files))}
ENVIRONMENT=environment_fingerprint()
RUNTIME={'python':sys.version.split()[0],'numpy':np.__version__,'kickmix':km.__version__}
REPORT_DIR=WORK/'reports';REPORT_DIR.mkdir(exist_ok=True)
def save_report(name,data):(REPORT_DIR/(name+'.json')).write_text(json.dumps(data,indent=2,default=str))
RESULTS={};PLAN={'invsqrt':'mrmeows 63Q, Alex Chilton v37 byte-pinned output','dqi':'Ahmed v8 Bezout-Schur splice, dataset v1','dfthcsos':'Ahmed v8 R4/B255/C128 factor, recompiled here at J5 from credited exact compiler'}
save_report('build',{'rows':BUILD,'seconds':time.monotonic()-t0,'members':MEMBERS})
print(f'15 exact J5 circuits rebuilt from source in {time.monotonic()-t0:.2f}s; all 18 member hashes PASS.',flush=True)


In [ ]:
import pandas as pd,matplotlib.pyplot as plt
from IPython.display import display,FileLink
FORCE_FULL=False
RECEIPT=json.loads(r'''{"reports":{"invsqrt":{"status":"PASS","score":108560.36455801228,"toffoli_equivalent":1723.1803898097187,"toffoli_count":1404.0939659609035,"max_toffoli_count":1490,"t_count":1.0,"rotation_count":44.0,"rotation_eps":1.8534217822524296e-13,"num_qubits":63,"reaction_depth":785,"num_ops":41252,"verified_shots":200540928,"worst_error_over_tolerance":0.9996626562985852,"worst_x":60,"circuit_sha256":"9ac50386192a544f353ab0f46604e5e072981b8bfc8521e8551c689a9445e3b9","wall_seconds":636.2630051360001},"dfthcsos":{"status":"PASS","score":426973779.3811362,"lambda_eff":47.48231835381439,"toffoli_count":20961.0,"num_qubits":429,"brot":8,"batch_size":5,"num_circuits":15,"reaction_depth":165,"num_ops":628250,"verified_shots":70144,"submission_sha256":"807534d81230861716d26f8f7643a4cb6a028a3f7856a44fab9442aaea17c37b","resources":{"brot":8,"keep_bits":10,"angles_toffoli_per_pass":4455.0,"angles_toffoli":8910.0,"angles_qubits":123,"angles_selection_qubits":11,"angles_persistent_qubits":40,"angles_temporary_qubits":72,"outer_state_prep_toffoli":378,"inner_state_prep_toffoli":9423,"givens_rotations_toffoli":2250,"outer_state_prep_persistent_qubits":42,"outer_state_prep_temporary_qubits":96,"outer_state_prep_qubits":138,"inner_state_prep_persistent_qubits":43,"inner_state_prep_temporary_qubits":304,"inner_state_prep_qubits":347,"heuristic_persistent_qubits":85,"heuristic_temporary_qubits":304,"heuristic_toffoli":12051,"heuristic_qubits":389,"persistent_qubits":125,"temporary_qubits":304,"toffoli":20961.0,"qubits":429,"angles_within_cap":true},"metrics":{"error_1":0.01891123452850644,"error_2":0.019997185000744346,"Lambda":289.8906208892117,"E_SOS":-22144.188444797765,"E_gap":3.9150947977650503,"lambda_eff":47.48231835381439,"gap_in_domain":true},"wall_seconds":28.394002893999982},"dqi":{"status":"PASS","score":487993123590.66656,"m":4095,"n":70,"degree":12,"r":2016,"verified_shots":102400,"value_errors":3,"phase_errors":0,"empirical_error_rate":2.9296875e-05,"effective_error_rate":0.000983909102617945,"log2_delta":-128.0,"repetition_factor":1.2055843238398525,"mean_error_weight":31.046737331746634,"toffoli_count":519152.2049902344,"max_toffoli_count":519157,"reaction_depth":10267,"num_ops":88933502,"shot_operation_counts":{"NEG":307173,"BIT_INVERT":250219315354,"BIT_STORE0":190260220212,"BIT_STORE1":674041161,"X":15968870400,"Z":13925754884,"R":7733678591,"HMR":46804886279,"CX":7084735234186,"CZ":116934385715,"SWAP":169239484878,"CSWAP":1262796800,"CCX":51898388991,"POP_CONDITION":271558348800,"PUSH_CONDITION":271558348800},"num_qubits":883,"num_bits":169120,"circuit_sha256":"569863ae0fafd5f94fe91b498ec67ed0a4d96b826ced31cb0ff57c7fe6c063cb","wall_seconds":426.736635654}},"members":{"dfthcsos/angle_k_0.kmb":"b631c3cafd2edc600edeed997f28c1b1b8a71234189bcc7d6a3afdf4a47408db","dfthcsos/angle_k_1.kmb":"974dc12398186f359e79d38d733720c5231686d7f591dcb1ac1b5d20be27f47e","dfthcsos/angle_k_10.kmb":"ac3cdb0d8d742d9629d3d5e3d5acb843012a084557587fe2c0ad9b9058da06d5","dfthcsos/angle_k_11.kmb":"9b103b5aadfc52e1b481894dff426dbf7443b5a3530db2c1e325ef6a39ba63e2","dfthcsos/angle_k_12.kmb":"d872682a1672429fdbbb44f56dbac9f4a0e951231817926d5470ff36f64aac3d","dfthcsos/angle_k_13.kmb":"42044bcb48d80947ef2900aea4eb2eaa8593f1b809d37bada48f1ab169b44ae3","dfthcsos/angle_k_14.kmb":"f3bb8ce79e39ecddad5bb1ff57e6818158c4cba1133b54f0009f1133b8585d49","dfthcsos/angle_k_2.kmb":"e25d78895511c50babe940b21314ede6ad79794d38a9da53efaefbdc802864ac","dfthcsos/angle_k_3.kmb":"ac59a43cf75a516937ad12900916c292add65558902df77b18d0600cc45a7ecb","dfthcsos/angle_k_4.kmb":"64d9dfc71e0fb4c5c55e4a01ba9063c579d17d2602551a0899253f8edd1dc136","dfthcsos/angle_k_5.kmb":"6c4b1e3a59e46a604528f80912f4ccfd9ce689c0f39cd548521d0663d9d1821a","dfthcsos/angle_k_6.kmb":"9932019836bbe8217a8fd34d1b54bbaccf9591d1c83f56c2a0cb20e2e14a2c7d","dfthcsos/angle_k_7.kmb":"7f1444666cf99cac8533089c862edfeb60db5e41daedf693c03af0d0be4b4392","dfthcsos/angle_k_8.kmb":"d748449d2c77cf858bdd90d69054bc3a3ac1d81ad2501c21d85e5b04a824e66b","dfthcsos/angle_k_9.kmb":"039ec2a8c81ff647a83832dc76a687947a80f81202361b9af2f1c6cef1e29268","dfthcsos/submission.npz":"a5eabd76c2b2869e39511a53130cd909034dec6f26fa3db61354a9d12ffe6413","dqi.kmb":"569863ae0fafd5f94fe91b498ec67ed0a4d96b826ced31cb0ff57c7fe6c063cb","invsqrt.kmb":"9ac50386192a544f353ab0f46604e5e072981b8bfc8521e8551c689a9445e3b9"},"environment":{"kickmix/kickmix.so":"c66b3216c4029e0cdfd09054306aa0feadf53ab2f50c1164e37f752874e5005e","verifiers/dfthcsos/FeMoCo76o113e.fcidump":"e5a48af746fd075026dec9e8f5df2d363802a121e8c53af9a0b984bde695e1bd","verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz":"7b7facf63fe35a9d9d3d00787ffdb700c863fa845fe4724d7f7c54778151cd87","verifiers/dfthcsos/__init__.py":"81915e324cc0cade67f52a54b7a51991fd96abaea94d1f8a4d423dae20e57348","verifiers/dfthcsos/example_submission/angle_k_reference_circuit.py":"8daccdca69f2d718b0176cd9281803df84a988ea5ac6c167fbb02e461ac2b371","verifiers/dfthcsos/scoreinfra/__init__.py":"be90e0d503d5c20dcac0102af638b9a4ccad0341daa5baa5d49216a259e4516f","verifiers/dfthcsos/scoreinfra/circuits.py":"fc027b52fdcebe14404f403a089122485d097ed65d03e6d989623d620b1ab68c","verifiers/dfthcsos/scoreinfra/costs_prepare.py":"7f1e190c42fc5325f3e6689fbbef51f95b0cb12f1c828cda92b6c088e72206da","verifiers/dfthcsos/scoreinfra/dfthc_math.py":"ae82898d47a7a86b96378fb906e0f8a9f46003065cf19c7c09952a123b5b5af8","verifiers/dfthcsos/scoreinfra/reference.py":"a2d5a63191d655729e29f0fde1e85fe4630bf896a91a63bb894d8d7daf71ed39","verifiers/dfthcsos/verify.py":"0775be7a528922367caebe9e4d051d6199811f1aefeaa4b793eaaa4621b8ec9d","verifiers/dqi/__init__.py":"01c870aa9763f4e4357fc6654bff1d3296cec995bd0943f87b4a48946ada5b40","verifiers/dqi/dqi_reference_circuit.py":"2b8e1b80a806157c333a7e308eaf2499ee09de499efbc05875202c327959959e","verifiers/dqi/verify_submission_dqi.py":"f14274bc0943fb9769ddb01131be6063191a15009fe1caf1d4bdd9fd25194e65","verifiers/invsqrt/__init__.py":"e9a6ecf253c343528578be5fa92eb820ccade63333aebb784552fb4ae75dbc0b","verifiers/invsqrt/nr_reference_circuit.py":"bb90330dcf774f7b502de471293facba67f27952cc8808fe4a9970a9b1322e23","verifiers/invsqrt/verify_submission_invsqrt.py":"ddef520dec6b68d1cf0a6f328ddd36d79eb84570b9192cf3f7acaf073f06733a"},"runtime":{"python":"3.13.15","numpy":"2.1.3","kickmix":"0.1"},"salt":"lightningv08-v13-fixed-20261026","projection":19.599952722197298,"plan":{"invsqrt":"mrmeows 63Q, Alex Chilton v37 byte-pinned output","dqi":"Ahmed v8 Bezout-Schur splice, dataset v1","dfthcsos":"Ahmed v8 R4/B255/C128 factor, recompiled here at J5 from credited exact compiler"},"baseline_chemistry":{"status":"PASS","score":432324941.69497436,"lambda_eff":47.48231835381439,"toffoli_count":21627.0,"num_qubits":421,"brot":8,"batch_size":4,"num_circuits":19,"reaction_depth":209,"num_ops":608394,"verified_shots":70144,"submission_sha256":"95dad220e9ec8a42cca4e410c7c4d7f36c828c7b6ddb71d6a39c5a6671ab222f","resources":{"brot":8,"keep_bits":10,"angles_toffoli_per_pass":4788.0,"angles_toffoli":9576.0,"angles_qubits":115,"angles_selection_qubits":11,"angles_persistent_qubits":32,"angles_temporary_qubits":72,"outer_state_prep_toffoli":378,"inner_state_prep_toffoli":9423,"givens_rotations_toffoli":2250,"outer_state_prep_persistent_qubits":42,"outer_state_prep_temporary_qubits":96,"outer_state_prep_qubits":138,"inner_state_prep_persistent_qubits":43,"inner_state_prep_temporary_qubits":304,"inner_state_prep_qubits":347,"heuristic_persistent_qubits":85,"heuristic_temporary_qubits":304,"heuristic_toffoli":12051,"heuristic_qubits":389,"persistent_qubits":117,"temporary_qubits":304,"toffoli":21627.0,"qubits":421,"angles_within_cap":true},"metrics":{"error_1":0.01891123452850644,"error_2":0.019997185000744346,"Lambda":289.8906208892117,"E_SOS":-22144.188444797765,"E_gap":3.9150947977650503,"lambda_eff":47.48231835381439,"gap_in_domain":true}}}''')
CACHE_VALID=(RECEIPT['members']==MEMBERS and RECEIPT['environment']==ENVIRONMENT and RECEIPT['runtime']==RUNTIME and RECEIPT['salt']==SALT)
EXPECTED_SHOTS={'invsqrt':200540928,'dqi':102400,'dfthcsos':70144}
MODES={}
for task in ['dfthcsos','invsqrt','dqi']:
    cached=RECEIPT['reports'].get(task,{})
    if task!='dfthcsos' and CACHE_VALID and not FORCE_FULL and cached.get('status')=='PASS' and cached.get('verified_shots')==EXPECTED_SHOTS[task]:
        r=cached;MODES[task]='Exact-environment full receipt + fresh probes'
    else:
        start=time.monotonic();print('Fresh full official verification:',task,flush=True)
        if task=='dfthcsos':r=verify_chemistry(FINAL/'dfthcsos',salt=SALT)
        elif task=='invsqrt':
            c=km.Circuit.from_file(str(FINAL/'invsqrt.kmb'),format='kmb')
            r=IV.verify(c,IV.ProblemSpecInvSqrt.for_n_p(10),salt=SALT,max_workers=WORKERS);del c
        else:
            c=km.Circuit.from_file(str(FINAL/'dqi.kmb'),format='kmb')
            r=DV.verify(c,DV.ProblemSpecDQI.benchmark_instance(),salt=SALT,max_workers=1);del c
        r['wall_seconds']=time.monotonic()-start;MODES[task]='Fresh full official run'
    assert r['status']=='PASS' and r['verified_shots']==EXPECTED_SHOTS[task]
    RESULTS[task]=r;save_report('full_'+task,r);gc.collect()
spec=IV.ProblemSpecInvSqrt.for_n_p(10)
c=km.Circuit.from_file(str(FINAL/'invsqrt.kmb'),format='kmb')
xreg,scratch=IV.validate_registers(c,spec)
rng=np.random.default_rng(20261026);sim=km.Simulator(batch_size=256,seed=20261026,debug_print=False)
for batch in range(64):
    values=rng.integers(1,spec.max_x+1,256,dtype=np.uint64)
    if batch==0:values[:7]=[1,2,3,4,32,spec.max_x-1,spec.max_x]
    sim.clear_for_shot();sim.write_across_shots(xreg,IV.to_bits(values,spec.width));sim.do(c)
    assert np.array_equal(IV.from_bits(sim.read_across_shots(xreg)),values)
    assert not sim.read_across_shots(scratch).any()
    phase=np.array([IV.half_turns_to_units(sim.read_shot_phase(i)) for i in range(256)],dtype=np.uint64)
    target,tol=IV.reference_values(spec,values)
    assert (np.abs((phase-target).view(np.int64)).view(np.uint64)<=tol).all()
del c,sim;gc.collect()
c=km.Circuit.from_file(str(FINAL/'dqi.kmb'),format='kmb')
fresh_dqi=DV.verify(c,DV.ProblemSpecDQI.benchmark_instance(num_shots=4096),salt=SALT+'-fresh',max_workers=1)
assert fresh_dqi['status']=='PASS';save_report('fresh_dqi',fresh_dqi);del c;gc.collect()
FULL_RECEIPT={**RECEIPT,'reports':RESULTS,'members':MEMBERS,'environment':ENVIRONMENT,'runtime':RUNTIME,'verification_modes':MODES}
(WORK/'full_verification.json').write_text(json.dumps(FULL_RECEIPT,indent=2))
display(pd.DataFrame([{'Task':k,'Status':v['status'],'Full shots':v['verified_shots'],'Evidence':MODES[k]} for k,v in RESULTS.items()]).style.hide(axis='index').format({'Full shots':'{:,.0f}'}))
plt.rcParams.update({'figure.dpi':140,'font.size':11,'axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold','text.color':'#16324b','axes.labelcolor':'#16324b','figure.facecolor':'white'})
COLORS={'invsqrt':'#27869c','dqi':'#7260a8','dfthcsos':'#be7d3c'}


## 2. Complete cost, not gate count alone

The competition combines the three reference-to-cost ratios geometrically:

$$S=\left(\prod_t C_{t,\mathrm{ref}}/C_t\right)^{1/3}.$$

The table uses full official verifier results. DQI includes its sampled failure penalty. The earlier C59 release used the same fixed salt, but the changed DQI circuit hash gives a different simulation seed; its cost comparison therefore includes sampling variation.

In [ ]:
OLD={'invsqrt':108560.36455801228,'dqi':511874506651.3508,'dfthcsos':433601590.02861047}
table=pd.DataFrame([{'Task':k,'Qubits':r['num_qubits'],'Mean Toffolis':r['toffoli_count'],'Cost':r['score'],'Reference / cost':REFERENCE[k]/r['score'],'Full shots':r['verified_shots']} for k,r in RESULTS.items()])
projection=math.prod(REFERENCE[k]/RESULTS[k]['score'] for k in REFERENCE)**(1/3)
display(table.style.hide(axis='index').format({'Cost':'{:,.3f}','Mean Toffolis':'{:,.1f}','Reference / cost':'{:.3f}x','Full shots':'{:,.0f}'}))
fig,ax=plt.subplots(figsize=(8.5,3.8),layout='constrained');keys=['invsqrt','dqi','dfthcsos'];x=np.arange(3)
ax.bar(x-.18,[REFERENCE[k]/OLD[k] for k in keys],.35,color='#d7e0e8',label='Previous C59 portfolio')
bars=ax.bar(x+.18,[REFERENCE[k]/RESULTS[k]['score'] for k in keys],.35,color=[COLORS[k] for k in keys],label='Selected portfolio')
for b in bars:ax.text(b.get_x()+b.get_width()/2,b.get_height()+.5,f'{b.get_height():.2f}x',ha='center')
ax.set(xticks=x,xticklabels=['Inverse square root','Bezout-Schur DQI','Chemistry'],ylabel='Reference / measured cost',title='The selected portfolio: full verifier measurements');ax.legend(frameon=False);ax.grid(axis='y',alpha=.15)
fig.savefig(WORK/'portfolio_gains.png',bbox_inches='tight');plt.show();table.to_csv(WORK/'verified_costs.csv',index=False)
print(f'Fixed-salt portfolio projection: {projection:.6f}')


## 3. The new idea: spend the free memory on a fifth word

After Ahmed's B=255 pruning, the lookup cache needs 72 temporary qubits. The address uses 11 qubits and each angle word uses 8 output qubits. In this cache layout,

$$Q_{\mathrm{ANGLE}}(J)=11+8J+72.$$

Thus J=4 needs 115 qubits, **J=5 fits at 123**, and J=6 would need 131 and violate the cap. Five also divides the 75 angle columns exactly: 15 batches cover every angle without a padded last word. This comparison concerns this cache layout, not a proof that every possible J=6 compiler is impossible.

Wider lookups make each circuit a little more expensive, while reducing the number of circuits. They also increase the persistent output register. The correct decision uses **the whole chemistry cost** $\lambda_{\mathrm{eff}}T_{\mathrm{total}}Q_{\mathrm{total}}$.

In [ ]:
BASELINE=RECEIPT['baseline_chemistry'];new=RESULTS['dfthcsos']
for key in ['error_1','error_2','Lambda','E_gap','lambda_eff']:assert abs(new['metrics'][key]-BASELINE['metrics'][key])<1e-9
ab=pd.DataFrame([{'Schedule':'Public J4 baseline','Words / batch':4,'ANGLE circuits':19,'Peak ANGLE qubits':115,'Whole qubits':BASELINE['num_qubits'],'Whole Toffolis':BASELINE['toffoli_count'],'Cost':BASELINE['score']},
 {'Schedule':'Source-built J5','Words / batch':5,'ANGLE circuits':15,'Peak ANGLE qubits':123,'Whole qubits':new['num_qubits'],'Whole Toffolis':new['toffoli_count'],'Cost':new['score']}])
display(ab.style.hide(axis='index').format({'Cost':'{:,.3f}','Whole Toffolis':'{:,.0f}'}))
q=new['num_qubits']/BASELINE['num_qubits'];t=new['toffoli_count']/BASELINE['toffoli_count'];ratio=new['score']/BASELINE['score']
fig,(ax,bx)=plt.subplots(1,2,figsize=(10.1,3.7),layout='constrained')
ax.bar(['J4 baseline','J5 wider lookup'],ab['Cost']/1e6,color=['#aebdca','#27869c']);ax.set(ylabel='Whole chemistry cost (millions)',title='Same physics, cheaper compilation');ax.set_ylim(0,470)
for i,v in enumerate(ab['Cost']/1e6):ax.text(i,v+4,f'{v:.3f}',ha='center')
grid=np.linspace(.98,1.05,200);bx.plot(grid,1/grid,color='#586474',label='Equal cost at fixed normalization');bx.fill_between(grid,.93,1/grid,alpha=.08,color='#27869c');bx.scatter([q],[t],s=80,color='#27869c',zorder=4)
bx.annotate(f'+{100*(q-1):.2f}% qubits\n{100*(t-1):.2f}% Toffolis',(q,t),xytext=(.985,.945),arrowprops={'arrowstyle':'->','color':'#586474'})
bx.set(xlabel='Whole qubit ratio: J5 / J4',ylabel='Whole Toffoli ratio: J5 / J4',title='More qubits can still lower cost',ylim=(.93,1.025));bx.legend(frameon=False,fontsize=9);bx.grid(alpha=.15)
fig.savefig(WORK/'qrom_cost_tradeoff.png',bbox_inches='tight');plt.show();ab.to_csv(WORK/'qrom_comparison.csv',index=False)
print(f'Cost ratio Q*T: {q:.6f} * {t:.6f} = {ratio:.6f}; chemistry reduction {100*(1-ratio):.4f}%')
print(f'Isolated portfolio gain: {100*(ratio**(-1/3)-1):.4f}%')
display(pd.DataFrame([{'J':4,'ANGLE qubits':115,'Fits 128':True},{'J':5,'ANGLE qubits':123,'Fits 128':True},{'J':6,'ANGLE qubits':131,'Fits 128':False}]).style.hide(axis='index'))


### Why the algebraic relation budget changes

The lookup emitter uses exact paired products over GF(2). One redundant sign bit leaves 7 independent Boolean functions per angle word. With 15 nonconstant low-address monomials, J4 has $15\cdot28=420$ mixed coordinates. Its 60 selected kernel relations leave 360 coordinates, represented by 180 paired products. The cache adds 72 Toffolis, giving **252 per circuit**.

J5 has $15\cdot35=525$ mixed coordinates. Retuning to **five relations per three-vertex block**, 75 independent relations overall, leaves 450 coordinates: **225 paired products + 72 cache Toffolis = 297 per circuit**. The exact tensor reconstruction is checked before emission.

Although 297 is larger than 252, $15\cdot297=4455$ is smaller than $19\cdot252=4788$. Counting forward and inverse ANGLE passes saves **666 Toffolis** in the complete block. Extra persistent output storage costs eight whole-system qubits. Both effects are included above. The factorization, integer words, normalization and Hamiltonian errors remain unchanged.

## 4. DQI: the public Bézout-Schur splice

Ahmed's v8 replaces the Schur amplitude-conversion middle block and its inverse with mrmeows's round17 central Bézout transform. Coefficient reversal, word swaps and CNOT basis changes adapt the representations without Toffolis. The classical mask projection also skips 70 scan points. The full decoder remains at 883 qubits and falls from roughly 536,798 to 519,152 mean Toffolis.

The supplied binary is reused unchanged and credited below. Rare failures still affect the repetition penalty. This curve is a **sensitivity scenario** using the official confidence-bound functions, not a leaderboard confidence interval. No salt is chosen from its result.

In [ ]:
d=RESULTS['dqi']
failures=np.arange(0,21)
scenario=[]
for count in failures:
    eps=DV.clopper_pearson_upper_bound(int(count),102400,log2_delta=-128)
    rep=DV.repetition_overhead(eps)
    cost=rep*d['num_qubits']**2*d['toffoli_count']
    score=(REFERENCE['dqi']/cost*REFERENCE['invsqrt']/RESULTS['invsqrt']['score']*REFERENCE['dfthcsos']/RESULTS['dfthcsos']['score'])**(1/3)
    scenario.append({'Failed shots':int(count),'Error upper bound':eps,'Repetition':rep,'Portfolio projection':score})
risk=pd.DataFrame(scenario)
observed=int(d['value_errors']+d['phase_errors'])
fig,ax=plt.subplots(figsize=(8.1,3.5),layout='constrained')
ax.plot(risk['Failed shots'],risk['Portfolio projection'],color='#217f9c',lw=2,marker='o',markersize=3)
if observed<=20:
    rr=risk.iloc[observed];ax.scatter([observed],[rr['Portfolio projection']],s=70,color='#bb7831',zorder=4,label=f'Full-check observation: {observed} failures');ax.legend(frameon=False)
ax.set(xlabel='Hypothetical failed shots out of 102,400',ylabel='Projected portfolio score',title='DQI failure sensitivity · all other costs held fixed')
ax.grid(alpha=.15);fig.savefig(WORK/'dqi_failure_sensitivity.png',bbox_inches='tight');plt.show()
display(risk.iloc[[0,1,2,5,10,20]].style.hide(axis='index').format({'Error upper bound':'{:.6f}','Repetition':'{:.5f}','Portfolio projection':'{:.6f}'}))
risk.to_csv(WORK/'dqi_failure_scenarios.csv',index=False)

## 5. What the other experiments taught us

The previous release developed offset-preserving ensemble coalescence, followed by a protected GPU refit. Its R10/C59 chemistry passed full verification and cost 433.602 million, 0.447% below its public C60 baseline. That archive received **19.24359** on the leaderboard. [Version 13 contains the derivation and research recipe](https://www.kaggle.com/code/lightningv08/schur-dqi-c59-circuit-cost-lab?scriptVersionId=362605535).

Applying the same merge to the new R4/C128 factorization, then refitting to C127, passed native physical accuracy but raised effective normalization to about 48.489. It lost on complete cost and is excluded. An earlier ordinary physical SVD also lost because its column orientation inflated the L1 normalization. These checks are useful: fewer factors or fewer qubits alone do not establish an improvement.

The selected update keeps Ahmed's newer coefficients unchanged and improves their exact lookup compilation. The fixed development salt is retained from the earlier release. Compiler search used exact algebra and native correctness checks; it did not select a favourable sampled DQI outcome.

## 6. Package the verified files

The official metric reader checks the archive contract. Every archived member is rehashed after compression and the ZIP CRC is checked. Outputs include full reports, exact compiler sources, search plans and the controlled comparison. Run All never submits automatically.

In [ ]:
_ = (WORK/"metric_preflight.py").write_text("\"\"\"Official Kaggle metric v10 readers, unchanged except formatting.\nSource: https://www.kaggle.com/code/metric/google-quantum-ai-metric\nAdapter preflight_directory only checks the file contract; task verification is separate.\n\"\"\"\nimport json\nimport math\nimport pathlib\nimport random\nimport stat\nimport sys\nimport time\nimport tokenize\nimport zipfile\nimport zlib\nfrom collections.abc import Callable, Mapping\nfrom typing import Any\nimport numpy as np\nimport pandas as pd\nALLOWED_INSTRUCTIONS = {'X', 'CX', 'CCX', 'Z', 'CZ', 'CCZ', 'Z_POW', 'NEG', 'SWAP', 'CSWAP', 'R', 'HMR', 'BIT_INVERT', 'BIT_STORE0', 'BIT_STORE1', 'PUSH_CONDITION', 'POP_CONDITION', 'INFO_TAG', 'DEBUG_PRINT'}\nMAX_QUBIT_OR_BIT_ID = 2 ** 30 // 1536\nMAX_NPZ_BYTES = 2 ** 26\n\nclass ParticipantVisibleError(Exception):\n    pass\n\ndef read_circuit(path: pathlib.Path, max_registers: int) -> Any:\n    \"\"\"Return a submitted .kmx or .kmb circuit as a kickmix.Circuit; raise ValueError unless it is fit to verify.\n\n    kickmix validates a file as it parses it, within limits that keep parsing cheap, but simulating\n    takes memory that grows with the largest qubit and bit ids, which kickmix allows up to ~5e8,\n    so those are capped at MAX_QUBIT_OR_BIT_ID. kickmix also allows a million registers, at\n    ~110 bytes each even if empty, so a circuit may have at most `max_registers`, the most its\n    benchmark needs. Every register must be named: the dfthcsos verifier crashes on an unnamed one.\n    \"\"\"\n    import kickmix\n    circuit = kickmix.Circuit.from_file(path, format=path.suffix.removeprefix('.'))\n    unsupported = set(circuit.operation_statistics()) - ALLOWED_INSTRUCTIONS\n    if unsupported:\n        raise ValueError(f'unsupported instructions {sorted(unsupported)}')\n    largest_id = max(circuit.num_qubits, circuit.num_bits) - 1\n    if largest_id >= MAX_QUBIT_OR_BIT_ID:\n        raise ValueError(f'qubit or bit id {largest_id} is above the limit of {MAX_QUBIT_OR_BIT_ID - 1}')\n    if circuit.num_registers > max_registers:\n        raise ValueError(f'{circuit.num_registers} registers, above the limit of {max_registers} for this benchmark')\n    if not all((isinstance(name, str) for name in circuit.register_data)):\n        raise ValueError('every register needs a name, given by its REGISTER line')\n    return circuit\n\ndef read_npz(path: pathlib.Path, spec: Mapping[str, tuple[str, tuple[int | str, ...]]]) -> dict[str, np.ndarray]:\n    \"\"\"Return the arrays `spec` names from an .npz; raise ValueError unless each is a plain number array as specified.\n\n    `spec` maps each key to the numpy dtype kinds it may have, such as 'f' for floats, and its shape,\n    in which a string names a dimension of any positive size that must agree wherever it recurs.\n\n    Each array's header is a Python literal of its dtype and shape, which np.load trusts: a few KB\n    of deflated zeros can claim a terabyte-sized array, and the dtype can be anything numpy can\n    describe, from structured and nested types to Python objects that need unpickling. So the\n    headers are read first, and the arrays loaded only if each is a plain number type of its kind,\n    not big-endian, with its shape, and they total at most MAX_NPZ_BYTES. Only format 1.0 headers,\n    which numpy.savez writes, are accepted, and numpy refuses any over 10 KB before parsing it.\n    \"\"\"\n    dimensions: dict[str, int] = {}\n    total = 0\n    with zipfile.ZipFile(path) as archive:\n        members = {name.removesuffix('.npy'): name for name in archive.namelist()}\n        missing = [key for key in spec if key not in members]\n        if missing:\n            raise ValueError(f'{path.name} has no {missing}')\n        for key, (kinds, expected_shape) in spec.items():\n            with archive.open(members[key]) as member:\n                if np.lib.format.read_magic(member) != (1, 0):\n                    raise ValueError(f'{key} in {path.name} is not in .npy format 1.0, which numpy.savez writes')\n                shape, _, dtype = np.lib.format.read_array_header_1_0(member)\n            if dtype.kind not in kinds or dtype.names is not None or dtype.subdtype is not None or (dtype.byteorder == '>'):\n                raise ValueError(f'{key} in {path.name} has dtype {dtype}, not a plain number type of numpy kind {kinds!r}')\n            matches = len(shape) == len(expected_shape) and all((size > 0 and size == (dimensions.setdefault(expected, size) if isinstance(expected, str) else expected) for size, expected in zip(shape, expected_shape, strict=True)))\n            if not matches:\n                raise ValueError(f'{key} in {path.name} has shape {shape}, not {expected_shape} with its named dimensions as in {dimensions}')\n            total += math.prod(shape) * dtype.itemsize\n    if total > MAX_NPZ_BYTES:\n        raise ValueError(f'the arrays in {path.name} take {total:,} bytes, above the limit of {MAX_NPZ_BYTES:,}')\n    with np.load(path, allow_pickle=False) as npz:\n        return {key: np.array(npz[key]) for key in spec}\n\ndef find_one(root: pathlib.Path, name: str, description: str, is_dir: bool=False) -> pathlib.Path:\n    \"\"\"The one path called `name` (a glob) under `root`, where the submission unpacked; raise ParticipantVisibleError otherwise.\"\"\"\n    paths = [path for path in root.rglob(name) if path.is_dir() == is_dir and '__MACOSX' not in path.relative_to(root).parts]\n    if len(paths) != 1:\n        raise ParticipantVisibleError(f'The submission must contain exactly one {description}, found {len(paths)}.')\n    return paths[0]\n\n\ndef preflight_directory(directory):\n    root = pathlib.Path(directory).resolve()\n    inv_path = find_one(root, 'invsqrt.km[xb]', 'inverse circuit')\n    dqi_path = find_one(root, 'dqi.km[xb]', 'DQI circuit')\n    chemistry = find_one(root, 'dfthcsos', 'chemistry directory', is_dir=True)\n    inv = read_circuit(inv_path, 1)\n    inv_qubits = inv.num_qubits\n    del inv\n    decoder = read_circuit(dqi_path, 4095 + 70)\n    decoder_qubits = decoder.num_qubits\n    del decoder\n    angles = sorted(chemistry.glob('angle_k_*.km[xb]'))\n    if not angles or {p.stem for p in angles} != {f'angle_k_{i}' for i in range(len(angles))}:\n        raise ValueError('Angle circuits must have consecutive numbering with no duplicate formats')\n    angle_widths = []\n    for path in angles:\n        circuit = read_circuit(path, 128)\n        if circuit.num_qubits > 128:\n            raise ValueError('ANGLE circuit exceeds the 128-qubit cap')\n        angle_widths.append(circuit.num_qubits)\n        del circuit\n    specification = {\n        'U': ('f', ('R', 'B', 76)), 'W': ('f', ('R', 'B', 'C')),\n        'H2_I': ('f', ('R', 'C')), 'H1_I': ('f', ()),\n        'Hsym_lower': ('f', (2926,)), 'tree_edges': ('iu', (75, 2))\n    }\n    npz_path = find_one(chemistry, 'submission.npz', 'factorization')\n    arrays = read_npz(npz_path, specification)\n    paths = [inv_path, dqi_path, *sorted(angles, key=lambda p: int(p.stem.rsplit('_', 1)[1])), npz_path]\n    return {'status': 'PASS', 'scope': 'Official file readers and archive layout; separate task PASS required',\n            'invsqrt_qubits': inv_qubits, 'dqi_qubits': decoder_qubits,\n            'angle_qubits': angle_widths, 'factorization_shape': list(arrays['W'].shape),\n            'archive_members': [str(p.relative_to(root)).replace('\\\\', '/') for p in paths]}\n",encoding="utf-8")

In [ ]:
from metric_preflight import preflight_directory
contract=preflight_directory(FINAL)
assert contract['status']=='PASS' and set(contract['archive_members'])==set(MEMBERS)
assert len(contract['angle_qubits'])==15 and max(contract['angle_qubits'])==123
archive=WORK/'submission.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for name in sorted(MEMBERS):
        info=zipfile.ZipInfo(name,date_time=(1980,1,1,0,0,0));info.create_system=3
        info.external_attr=0o600<<16;info.compress_type=zipfile.ZIP_DEFLATED
        with z.open(info,'w') as target,(FINAL/name).open('rb') as source:shutil.copyfileobj(source,target,8<<20)
with zipfile.ZipFile(archive) as z:
    assert len(z.namelist())==len(set(z.namelist()))==18 and z.testzip() is None
    assert {n:hashlib.file_digest(z.open(n),'sha256').hexdigest() for n in z.namelist()}==MEMBERS
manifest={'members':MEMBERS,'archive_sha256':sha(archive),'archive_bytes':archive.stat().st_size,'contract':contract,
 'projection':projection,'full_environment_match':CACHE_VALID,'verification_modes':MODES,'fixed_development_salt':SALT,
 'previous_observed_score':19.24359,'current_version_observed_score':None,'build':BUILD}
(WORK/'release_manifest.json').write_text(json.dumps(manifest,indent=2))
table.to_csv(WORK/'verified_costs.csv',index=False)
(WORK/'LICENSE').write_text(COMPILER_SOURCE['LICENSE'])
(WORK/'NOTICE').write_text(COMPILER_SOURCE['NOTICE']+'\n'+(DATA/'DQI_NOTICE.md').read_text())
print(f'submission.zip: {archive.stat().st_size/2**20:.2f} MiB | 18 verified members')
print('SHA-256:',manifest['archive_sha256']);print(f'Fixed-salt full-check projection: {projection:.6f}')
display(FileLink(str(archive)))


## Sources and credit

- [Ahmed Berat Özer — DF-THC Group Compression, v8](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), [v8 components](https://www.kaggle.com/datasets/ahmedberatozer/vqcc-v8-components): the Bézout–Schur splice, R4/B255/C128 factorization, pruning/quantization/offset optimization, and generalized kernel-relation compiler. The DQI binary and factorization are unchanged here.
- [mrmeows — open circuits and research notes](https://www.kaggle.com/code/mrmeows/15-75-circuits-let-s-all-reach-the-plateau): the 63-qubit inverse square root, round17 Bézout transform, paired-product lookup algebra, kernel tools, native checker and HMR emitter. Compiler sources are Apache 2.0.
- [Alex Chilton — Schur at 883 qubits](https://www.kaggle.com/code/alexchilton/toffolis-go-18-3-schur-dqi-at-883-qubits): the narrower reverse synthesis and the pinned invsqrt source output. The continued-fraction Schur design originates with **jday96314**. The DQI source notice is preserved verbatim; its upstream repository does not state a source-code licence.
- Earlier foundations credited upstream include Dmitrii Gluzdov, SomeJoint / Agniv Banerjee, manwithacat, Phuc Thai, Kozykorpesh, Artem1410, Rudra Bhaskar, Marsyas, Sumit Chongder, marculera, Michael Rustad and other contributors named in the source headers and DQI notice.
- [Google Quantum AI competition data and verifiers](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/data) define the register contracts, accuracy constraints and cost; they are unchanged.

New here: use the freed lookup workspace for five words, retune the relation budget, regenerate the exact certificates and compare full costs under unchanged physics. Earlier C59 experiments remain available in history. The compiler sources, plans, licence and attribution notices are exported with the result.